# 02 — Reproduce figure and tables.

**Start here: run all cells in order from this folder.** Keep `RUN_SCORING = False` in `01_scoring.ipynb`; the completed measurements are supplied in `data/scored_data.zip`. This notebook needs no API key and runs offline.

The notebook recomputes all results.

Inputs are `data/raw_data.zip` (the optional scoring workflow) and `data/scored_data.zip` (this analysis). Leave both archives zipped. Intermediate files and embedded Python modules are created in a temporary workspace; set `WORK_DIR` below to an empty external directory only if you want to retain them. The completed notebook also retains its previews and final verification report.

In [ ]:
from pathlib import Path
import os,sys,json,tempfile,zipfile,hashlib,urllib.request,warnings,importlib.metadata,platform,time
from datetime import datetime,timezone
ANALYSIS_STARTED = time.perf_counter()
ANALYSIS_STARTED_UTC = datetime.now(timezone.utc).isoformat()
import numpy as np,pandas as pd
from IPython.display import display,Image,Markdown,HTML

PACKAGE_DIR = None  # Leave None when this notebook is launched from its folder.
WORK_DIR = None     # Optional external folder for retaining intermediate files.
requested=PACKAGE_DIR or os.environ.get('WORDS_TRAVEL_PACKAGE')
start=Path.cwd().resolve()
def is_package(path):
    path=Path(path)
    n=path/'02_analysis.ipynb'
    if not n.is_file() or not (path/'data/scored_data.zip').is_file():return False
    return json.loads(n.read_text()).get('metadata',{}).get('words_travel',{}).get('package_id')=='words-travel-el-sep28-v9'
candidates=[Path(requested).expanduser()] if requested else [start,*start.parents,globals().get('PACKAGE',start)]
PACKAGE=next((Path(p).resolve() for p in candidates if is_package(p)),None)
if PACKAGE is None:raise FileNotFoundError('Open this notebook from its folder, or set PACKAGE_DIR to the V9 folder.')
_temporary=tempfile.TemporaryDirectory(prefix='words_travel_analysis_') if WORK_DIR is None else None
ROOT=Path(_temporary.name if _temporary else WORK_DIR).expanduser().resolve()
if ROOT==PACKAGE or PACKAGE in ROOT.parents:raise ValueError('WORK_DIR must be outside this manuscript folder.')
if WORK_DIR is not None and ROOT.exists() and any(ROOT.iterdir()):raise ValueError('Choose an empty WORK_DIR for a fresh run.')
ROOT.mkdir(parents=True,exist_ok=True)
for name in ['code','data','derived','results','output','figures','audit']:(ROOT/name).mkdir(exist_ok=True)
with zipfile.ZipFile(PACKAGE/'data/scored_data.zip') as archive:
    manifest=json.loads(archive.read('manifest.json'))
    assert manifest['kind']=='words-travel-scoring-output' and manifest['schema_version']==1
    assert set(archive.namelist())==set(manifest['files'])|{'manifest.json'}
    for name,expected in manifest['files'].items():
        destination=(ROOT/name).resolve()
        if not destination.is_relative_to(ROOT):raise ValueError('Unsafe archive member: '+name)
        data=archive.read(name)
        assert hashlib.sha256(data).hexdigest()==expected,('Scoring-output checksum mismatch',name)
        destination.parent.mkdir(parents=True,exist_ok=True);destination.write_bytes(data)
# No scoring or API client is imported. Disable HTTP as an additional guarantee.
def offline(*args,**kwargs):raise RuntimeError('This analysis is offline; network access is disabled.')
urllib.request.urlopen=offline
try:
    import requests
    requests.sessions.Session.request=offline
except ImportError:pass
os.environ['MPLCONFIGDIR']=str(ROOT/'matplotlib-cache')
for name in ['definitions','data','estimation','diagnostics','analysis','measurement_checks','supplementary','granularity_ties','attention_events','representation_diagnostics','inference_sensitivity','identification','saved_measurements','verification','exhibits','figures','publication']:
    sys.modules.pop(name,None)
os.chdir(ROOT);sys.path.insert(0,str(ROOT/'code'))
warnings.filterwarnings('ignore',category=pd.errors.PerformanceWarning)
pd.set_option('display.max_rows',60);pd.set_option('display.max_columns',12)
versions={name:importlib.metadata.version(name) for name in ['numpy','pandas','scipy','scikit-learn','statsmodels','matplotlib','openpyxl','ipython','pillow']}
print('Python:',platform.python_version(),'\nPackages:',versions)
print('Verified saved measurement files:',len(manifest['files']))
print('Temporary analysis workspace:',ROOT)


## Calculation code


### Definitions

In [ ]:
%%writefile code/definitions.py
"""Fixed economic definitions and country classifications; no fitted results."""
SOURCE_OFFICIAL_ENGLISH = {'AU','CA','SZ','ECB','GM','GH','IS','IN','KE','LS','LR','MW','NZ','NG','PK','PH','RW','SL','ZA','UG','GB','US','ZM'}
ADVANCED = {'AU', 'CA', 'CH', 'CZ', 'ECB', 'GB', 'IL', 'IS', 'JP', 'KR', 'NO', 'NZ', 'SE', 'TW', 'US'}
ENGLISH_ORIGINAL = {'US', 'GB', 'CA', 'AU', 'NZ', 'ECB'}
INFLATION_TARGETERS = {'AU', 'BR', 'CA', 'CL', 'CO', 'CZ', 'GB', 'GH', 'HU', 'ID', 'IL', 'IN', 'IS', 'KR', 'MX', 'NZ', 'NO', 'PH', 'PL', 'SE', 'TH', 'TR', 'ZA'}
ISO2_TO_ISO3 = {'AR': 'ARG', 'AU': 'AUS', 'BR': 'BRA', 'CA': 'CAN', 'CH': 'CHE', 'CL': 'CHL', 'CO': 'COL', 'CZ': 'CZE', 'EG': 'EGY', 'GB': 'GBR', 'GH': 'GHA', 'GM': 'GMB', 'HU': 'HUN', 'ID': 'IDN', 'IL': 'ISR', 'IN': 'IND', 'IS': 'ISL', 'JP': 'JPN', 'KE': 'KEN', 'KR': 'KOR', 'KZ': 'KAZ', 'LK': 'LKA', 'LR': 'LBR', 'LS': 'LSO', 'MD': 'MDA', 'MN': 'MNG', 'MW': 'MWI', 'MX': 'MEX', 'MY': 'MYS', 'MZ': 'MOZ', 'NG': 'NGA', 'NO': 'NOR', 'NZ': 'NZL', 'PH': 'PHL', 'PK': 'PAK', 'PL': 'POL', 'RW': 'RWA', 'SE': 'SWE', 'SL': 'SLE', 'SZ': 'SWZ', 'TH': 'THA', 'TJ': 'TJK', 'TR': 'TUR', 'TW': 'TWN', 'UA': 'UKR', 'UG': 'UGA', 'US': 'USA', 'UZ': 'UZB', 'ZA': 'ZAF', 'ZM': 'ZMB'}
COUNTRY_NAME = {'AR': 'Argentina', 'AU': 'Australia', 'BR': 'Brazil', 'CA': 'Canada', 'CH': 'Switzerland', 'CL': 'Chile', 'CO': 'Colombia', 'CZ': 'Czechia', 'ECB': 'Euro area', 'EG': 'Egypt', 'GB': 'United Kingdom', 'GH': 'Ghana', 'GM': 'Gambia', 'HU': 'Hungary', 'ID': 'Indonesia', 'IL': 'Israel', 'IN': 'India', 'IS': 'Iceland', 'JP': 'Japan', 'KE': 'Kenya', 'KR': 'Korea', 'KZ': 'Kazakhstan', 'LK': 'Sri Lanka', 'LR': 'Liberia', 'LS': 'Lesotho', 'MD': 'Moldova', 'MN': 'Mongolia', 'MW': 'Malawi', 'MX': 'Mexico', 'MY': 'Malaysia', 'MZ': 'Mozambique', 'NG': 'Nigeria', 'NO': 'Norway', 'NZ': 'New Zealand', 'PH': 'Philippines', 'PK': 'Pakistan', 'PL': 'Poland', 'RW': 'Rwanda', 'SE': 'Sweden', 'SL': 'Sierra Leone', 'SZ': 'Eswatini', 'TH': 'Thailand', 'TJ': 'Tajikistan', 'TR': 'Turkey', 'TW': 'Taiwan', 'UA': 'Ukraine', 'UG': 'Uganda', 'US': 'United States', 'UZ': 'Uzbekistan', 'ZA': 'South Africa', 'ZM': 'Zambia'}
CONCEPTS = ['activity_outlook', 'inflation_outlook', 'hawkish_reaction', 'supply_narrative', 'uncertainty', 'fx_pressure', 'financial_stability']
HF_CONCEPTS = CONCEPTS[:5] + ['disagreement']
LABELS = {'activity_outlook': 'Activity outlook', 'inflation_outlook': 'Inflation outlook', 'hawkish_reaction': 'Hawkish language', 'supply_narrative': 'Supply attribution', 'uncertainty': 'Uncertainty', 'fx_pressure': 'Exchange-rate pressure', 'financial_stability': 'Financial stability', 'disagreement': 'Disagreement'}
LM_NEG = {'loss', 'losses', 'decline', 'declined', 'negative', 'default', 'litigation', 'impairment', 'volatile', 'weakness', 'adverse', 'failure', 'unfavorable', 'downturn', 'recession', 'crisis', 'risk', 'risks', 'uncertain', 'uncertainty', 'deterioration', 'shortage'}
LM_POS = {'gain', 'gains', 'improve', 'improved', 'strong', 'strength', 'profitable', 'opportunity', 'favorable', 'stable', 'confidence', 'expansion', 'robust', 'solid', 'resilient'}
HAWKISH_WORDS = {'tighten', 'tightening', 'restrictive', 'hike', 'hikes', 'hawkish', 'firming', 'raise', 'raising', 'increase', 'increasing', 'higher', 'hot', 'overheating', 'remove', 'accommodation', 'restrict', 'restricting'}
DOVISH_WORDS = {'ease', 'easing', 'accommodative', 'cut', 'cuts', 'dovish', 'lower', 'decrease', 'decreasing', 'patient', 'stimulus', 'supportive', 'loosen', 'loosening'}


### Data

In [ ]:
%%writefile code/data.py
"""Build all analysis inputs from raw files; benchmark files are audit inputs only."""
from pathlib import Path
import re, json, hashlib, unicodedata
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from definitions import *

ROOT = Path(__file__).resolve().parents[1]
CONCEPT_ALL = CONCEPTS + ['disagreement']
def sha(s): return hashlib.sha256(s.encode('utf8')).hexdigest()
def read_jsonl(path):
    # JSON strings may legally contain Unicode paragraph separators. Only actual
    # newline bytes delimit JSONL records; str.splitlines() is too broad.
    with Path(path).open(encoding='utf8') as f:return [json.loads(line) for line in f if line.strip()]
def clean(raw):
    s = raw.replace('\r','\n')
    m = re.search(r'\ntext:\s*',s,re.I)
    if m and m.start()<800: s=s[m.end():]
    s=re.sub(r'https?://\S+',' ',s)
    for p in ['url:','main title:','sub title:','location:']:
        s=re.sub(r'^'+p+r'.*$',' ',s,flags=re.I|re.M)
    return re.sub(r'\s+',' ',s.replace('Share',' ')).strip()
def read_text(p):
    b=p.read_bytes()
    try: return b.decode('utf-8-sig')
    except UnicodeDecodeError: return b.decode('cp1252')
def file_date(p):
    m=re.search(r'(\d{4})[_-]?(\d{2})[_-]?(\d{2})',p.stem)
    if not m:return None
    return pd.to_datetime('-'.join(m.groups()),errors='coerce')
def norm_quote(s):
    s=unicodedata.normalize('NFKC',str(s)).replace('’',"'").replace('‘',"'").replace('“','"').replace('”','"')
    return re.sub(r'\s+',' ',s).strip().casefold()
def quote_present(q,t): return bool(str(q).strip()) and norm_quote(q) in norm_quote(t)
def z(s):
    sd=s.std(ddof=0)
    return (s-s.mean())/sd if pd.notna(sd) and sd>1e-12 else s*0
def load_raw():
    rows=[]; flow=[]
    for p in sorted((ROOT/'data/mpsd_raw').glob('*/*.txt')):
        country=p.parent.name;dt=file_date(p);t=clean(read_text(p))
        reason='retained'
        if pd.isna(dt) or dt is None:reason='no_valid_filename_date'
        elif len(t)<80:reason='below_original_80_character_filter'
        flow.append(dict(path=str(p.relative_to(ROOT)),country=country,date=str(dt),characters=len(t),status=reason))
        if reason!='retained':continue
        tokens=re.findall(r"[A-Za-z']+",t.lower())[:250]
        stops={'the','of','and','to','in','a','is','that','for','on','with','as','by','this','are','was','be','it','from','or','an','at','which','have','has','not','will','been','their','its'}
        rows.append(dict(doc_id=f'{country}_{dt:%Y-%m-%d}',country=country,date=dt,year=dt.year,text=t,
            word_count=len(re.findall(r'[A-Za-z]+',t)),characters=len(t),text_hash=sha(t),
            source_path=str(p.relative_to(ROOT)),advanced=int(country in ADVANCED),iso3=ISO2_TO_ISO3.get(country),
            country_name=COUNTRY_NAME.get(country,country),english_original=int(country in ENGLISH_ORIGINAL),
            inflation_targeter=int(country in INFLATION_TARGETERS),en_share=sum(w in stops for w in tokens)/max(1,len(tokens))))
    d=pd.DataFrame(rows).sort_values(['country','date','source_path'])
    d[d.duplicated(['country','date'],keep=False)].drop(columns='text').to_csv(ROOT/'audit/duplicate_country_dates.csv',index=False)
    d=d.drop_duplicates(['country','date']).reset_index(drop=True)
    d['duplicate_text']=d.duplicated(['country','text_hash'],keep=False)
    d['likely_english']=(d.en_share>=.10).astype(int);d['log_words']=np.log(d.word_count.clip(lower=1))
    d['official_english']=d.country.isin(SOURCE_OFFICIAL_ENGLISH).astype(int)
    d['translation_status']=np.where(d.official_english.eq(1),'Official English','Database translated')
    pd.DataFrame(flow).to_csv(ROOT/'audit/corpus_screen.csv',index=False)
    d.to_json(ROOT/'derived/corpus.jsonl',orient='records',lines=True,date_format='iso',force_ascii=False)
    return d
def load_external_text(folder,bank):
    rows=[]
    for p in sorted((ROOT/'data/external'/folder).glob('*.txt')):
        dt=file_date(p);t=clean(read_text(p))
        if dt is None or pd.isna(dt) or len(t)<80:continue
        rows.append(dict(doc_id=f'{bank}_{folder}_{dt:%Y-%m-%d}',country=bank,date=dt,year=dt.year,text=t,
            word_count=len(re.findall(r'[A-Za-z]+',t)),characters=len(t),text_hash=sha(t),source_path=str(p.relative_to(ROOT))))
    return pd.DataFrame(rows).drop_duplicates('date').sort_values('date').reset_index(drop=True)
def fit_lexical(library):
    texts=[]; poles={}
    for c in library:
        for p in ['positive','negative']:
            start=len(texts);texts+=library[c][p];poles[c,p]=slice(start,len(texts))
    v=TfidfVectorizer(stop_words='english',ngram_range=(1,2),min_df=1,sublinear_tf=True,norm='l2')
    a=v.fit_transform(texts)
    weights={c:np.asarray(a[poles[c,'positive']].mean(axis=0)-a[poles[c,'negative']].mean(axis=0)).ravel() for c in library}
    return v,weights
def lexical(texts,v,w):
    a=v.transform(texts)
    return pd.DataFrame({c:np.asarray(a@weight).ravel() for c,weight in w.items()})
def score_lexical(d):
    lib=json.loads((ROOT/'data/exemplars.json').read_text());v,w=fit_lexical(lib)
    x=lexical(d.text,v,w);x.index=d.index
    out=pd.concat([d,x],axis=1)
    out.drop(columns='text').to_csv(ROOT/'derived/lexical_scores.csv',index=False)
    return out,v,w
def source_screens(d):
    months='January|February|March|April|May|June|July|August|September|October|November|December'
    pat=re.compile(rf'(?:({months})\s+\d{{1,2}}(?:st|nd|rd|th)?[,]?\s+\d{{4}}|\d{{1,2}}\s+({months})\s+\d{{4}})',re.I)
    rows=[]
    for r in d.itertuples():
        m=pat.search(r.text[:500])
        if m:
            dt=pd.to_datetime(m.group(),errors='coerce')
            if pd.notna(dt) and abs((dt-r.date).days)>14:
                rows.append(dict(doc_id=r.doc_id,country=r.country,filename_date=r.date,heading_date=dt,gap_days=(dt-r.date).days,status='candidate_requires_source_review',matched_heading=m.group()))
    flags=pd.DataFrame(rows);flags.to_csv(ROOT/'audit/heading_date_candidates.csv',index=False)
    return flags
def content_ngrams(text):
    # Contiguous original words; never bridge removed stopwords. At least two content words.
    toks=re.findall(r'[a-z]+',text.lower());out=set()
    for n in [3,4]:
        for i in range(len(toks)-n+1):
            chunk=toks[i:i+n]
            if sum(t not in ENGLISH_STOP_WORDS for t in chunk)>=2:
                out.add(' '.join(chunk))
    return out
def make_pairs(d,source):
    import difflib
    rows=[]
    for bank,g in d.groupby('country'):
        g=g.sort_values('date')
        for prev,curr in zip(g.iloc[:-1].itertuples(),g.iloc[1:].itertuples()):
            a=prev.text.split();b=curr.text.split();sm=difflib.SequenceMatcher(None,a,b,autojunk=False)
            parts=[];edits=[]
            for tag,i,j,k,l in sm.get_opcodes():
                if tag=='equal':parts.append(' '.join(a[i:j]))
                else:
                    if i<j:parts.append('[-'+ ' '.join(a[i:j])+'-]')
                    if k<l:parts.append('{+'+' '.join(b[k:l])+'+}')
                    edits.append(dict(operation=tag,old_start=i,old_end=j,new_start=k,new_end=l,old=' '.join(a[i:j]),new=' '.join(b[k:l])))
            rows.append(dict(pair_id=f'{source}_{bank}_{curr.date:%Y-%m-%d}',country=bank,source=source,date=str(curr.date.date()),previous_date=str(prev.date.date()),
                previous_doc_id=prev.doc_id,doc_id=curr.doc_id,old_text=prev.text,new_text=curr.text,redline=' '.join(parts),edits=edits,
                old_text_hash=sha(prev.text),text_hash=sha(curr.text),redline_hash=sha(' '.join(parts)),word_similarity=sm.ratio(),identical=prev.text==curr.text))
    return rows
if __name__=='__main__':
    raw=load_raw();sc,v,w=score_lexical(raw);flags=source_screens(raw)
    intro=load_external_text('ecb_intro','ECB');intro.to_json(ROOT/'derived/ecb_intro_corpus.jsonl',orient='records',lines=True,date_format='iso')
    pairs=make_pairs(raw[raw.country.isin(['US','ECB'])],'mpsd')+make_pairs(intro,'intro')
    with (ROOT/'audit/leader_redlines.jsonl').open('w') as f:
        for row in pairs:f.write(json.dumps(row,ensure_ascii=False)+'\n')
    print('Corpus',len(raw),'jurisdictions',raw.country.nunique(),'raw files',len(pd.read_csv(ROOT/'audit/corpus_screen.csv')),'heading candidates',len(flags))
    print('Pairs',pd.Series([p['source']+'_'+p['country'] for p in pairs]).value_counts().to_dict())


### Estimation

In [ ]:
%%writefile code/estimation.py
"""Fresh OLS/WLS, multiway inference, contrasts and explicit unidentified terms."""
import numpy as np
import pandas as pd
from scipy import stats, linalg
from statsmodels.stats.multitest import multipletests
from itertools import combinations

def bh(df,groups,p='p_value',q='q_value',family_size=None):
    df=df.copy();df[q]=np.nan
    chunks=df.groupby(groups,dropna=False) if groups else [(None,df)]
    for _,g in chunks:
        idx=g.index[g[p].notna()];v=df.loc[idx,p].to_numpy()
        if not len(v):continue
        m=family_size or len(g)
        df.loc[idx,q]=multipletests(np.r_[v,np.ones(max(0,m-len(v)))],method='fdr_bh')[1][:len(v)]
    return df
def cluster_meat(scores,labels):
    code=pd.factorize(labels)[0];n=code.max()+1
    sums=np.zeros((n,scores.shape[1]));np.add.at(sums,code,scores)
    return sums.T@sums,n
def fit(d,y,x,fe=('country','year'),clusters=('country','fed_date'),weight=None,contrasts=None):
    needed=list(dict.fromkeys([y,*x,*fe,*clusters]+([weight] if weight else [])))
    a=d[needed].replace([np.inf,-np.inf],np.nan).dropna().copy()
    meta=dict(n=len(a),n_countries=a.country.nunique() if 'country' in a else np.nan)
    if len(a)<10:
        names=list(x)+list((contrasts or {}).keys())
        return [dict(term=t,estimate=np.nan,std_error=np.nan,p_value=np.nan,ci_low=np.nan,ci_high=np.nan,
                     status='insufficient_observations',covariance_valid=False,n_events=a[clusters[1]].nunique() if len(clusters)>1 else np.nan,
                     df_t=np.nan,rank=0,parameters=0,cluster_dimensions=';'.join(clusters),r_squared=np.nan,**meta) for t in names],None
    varying=[c for c in x if a[c].std(ddof=0)>1e-12 or (c=='intercept' and a[c].eq(1).all())]
    constant=[] if 'intercept' in varying else [pd.DataFrame({'const':np.ones(len(a))},index=a.index)]
    X=pd.concat([*constant,a[varying].astype(float),*[pd.get_dummies(a[c],prefix=c,drop_first=True,dtype=float) for c in fe]],axis=1)
    # Fit in full design to retain finite sample cluster correction for absorbed effects.
    w=np.ones(len(a)) if weight is None else a[weight].to_numpy(float)
    xx=X.to_numpy(float)*np.sqrt(w)[:,None];yy=a[y].to_numpy(float)*np.sqrt(w)
    beta,_,rank,_=linalg.lstsq(xx,yy,lapack_driver='gelsy');res=yy-xx@beta
    bread=linalg.pinvh(xx.T@xx);scores=xx*res[:,None]
    cov=np.zeros_like(bread);cluster_counts=[]
    if clusters:
        for size in range(1,len(clusters)+1):
            for subset in combinations(clusters,size):
                labels=pd.MultiIndex.from_frame(a[list(subset)])
                meat,g=cluster_meat(scores,labels)
                if size==1:cluster_counts.append(g)
                factor=g/max(1,g-1)*(len(a)-1)/max(1,len(a)-X.shape[1])
                cov+=(-1)**(size+1)*factor*bread@meat@bread
        dof=min(cluster_counts)-1
    else:
        h=np.sum(xx*(xx@bread),axis=1);s=xx*(res/np.clip(1-h,1e-8,None))[:,None]
        cov=bread@s.T@s@bread;dof=np.inf
    if not (np.isfinite(beta).all() and np.isfinite(cov).all() and np.isfinite(res).all()):
        raise ArithmeticError('Non-finite regression calculation; no result accepted')
    weighted_mean=np.average(a[y].to_numpy(float),weights=w)
    tss=np.sum(w*(a[y].to_numpy(float)-weighted_mean)**2)
    meta.update(n_events=cluster_counts[1] if len(cluster_counts)>1 else np.nan,df_t=dof,rank=int(rank),parameters=X.shape[1],cluster_dimensions=';'.join(clusters),r_squared=1-res@res/max(1e-30,tss))
    rows=[]
    def record(name,vec,estimable=True):
        b=float(vec@beta);variance=float(vec@cov@vec);valid=variance>0 and estimable and dof>0
        se=np.sqrt(variance) if valid else np.nan
        critical=stats.t.ppf(.975,dof) if np.isfinite(dof) else stats.norm.ppf(.975)
        pv=2*stats.t.sf(abs(b/se),dof) if valid else np.nan
        rows.append(dict(term=name,estimate=b if estimable else np.nan,std_error=se,p_value=pv,ci_low=b-critical*se,ci_high=b+critical*se,
            status='identified' if estimable else 'unidentified',covariance_valid=valid,**meta))
    for term in x:
        vec=np.zeros(len(beta))
        if term in X:vec[X.columns.get_loc(term)]=1
        record(term,vec,term in X and rank==X.shape[1])
    for name,weights in (contrasts or {}).items():
        vec=np.zeros(len(beta));ok=all(t in X for t in weights)
        if ok:
            for t,v in weights.items():vec[X.columns.get_loc(t)]=v
        record(name,vec,ok and rank==X.shape[1])
    return rows,dict(data=a,X=X,beta=beta,cov=cov,resid=res,bread=bread,df_t=dof)

def add_method(d,values,prefix):
    from data import z,CONCEPTS
    d=d.copy()
    for c in CONCEPTS:
        col=prefix+c;d[col]=values[c].to_numpy()
        d['z_'+col]=d.groupby('country')[col].transform(z)
        diff=d.groupby('country')[col].diff()
        d['zd_'+col]=diff.groupby(d.country).transform(z)
    return d
def match_leaders(d,prefixes,ecb=None):
    from data import CONCEPTS
    d=d.sort_values('date');f=d[~d.country.isin(['US','ECB'])].copy()
    dates=set(d.loc[d.country.isin(['US','ECB']),'date'])
    f=f[~f.date.isin(dates)]
    cols=['zd_'+p+c for p in prefixes for c in CONCEPTS]
    for code,pre,direction in [('US','fed','backward'),('ECB','ecb','backward'),('US','future','forward')]:
        leader=(ecb if code=='ECB' and ecb is not None else d[d.country==code])
        ev=leader[['date',*cols]].copy()
        ev[pre+'_date']=ev.date
        ev=ev.rename(columns={c:pre+'_'+c for c in cols})
        f=pd.merge_asof(f.sort_values('date'),ev.sort_values('date'),on='date',direction=direction,allow_exact_matches=False)
    for p in ['fed','ecb']:f['days_since_'+p]=(f.date-f[p+'_date']).dt.days
    f['days_until_fed']=(f.future_date-f.date).dt.days
    f['union45']=f.days_since_fed.between(1,45)|f.days_since_ecb.between(1,45)
    return f


### Diagnostics

In [ ]:
%%writefile code/diagnostics.py
"""A1/A2/A3/A6/B1/B2, recomputed with raw corpus and audit-only old caches."""
from data import *
from estimation import *
def run():
    raw=pd.read_json(ROOT/'derived/corpus.jsonl',lines=True);raw['date']=pd.to_datetime(raw.date)
    lex=pd.read_csv(ROOT/'derived/lexical_scores.csv',parse_dates=['date'])
    old=pd.read_json(ROOT/'data/benchmark/llm_direct_scores.jsonl',lines=True);old.date=pd.to_datetime(old.date)
    emb=pd.read_csv(ROOT/'data/benchmark/embedding_scores.csv',parse_dates=['date'])
    d=raw.merge(lex[['country','date',*CONCEPTS]],on=['country','date'],validate='one_to_one').merge(old,on=['country','date'],validate='one_to_one').merge(emb,on=['country','date'],validate='one_to_one').sort_values(['country','date']).reset_index(drop=True)
    rows=[];mass=[];distribution=[];rates=[]
    for c in CONCEPTS:
        vc=d['llm_'+c].value_counts()
        mass.append(dict(concept=c,n=len(d),mass_at_4=vc.get(4,0)/len(d),unique_values=len(vc)))
        for country,g in d.groupby('country'):
            for value,n in g['llm_'+c].value_counts().items():distribution.append(dict(country=country,concept=c,rating=value,n=n,share=n/len(g)))
            for method,p in [('TF-IDF',''),('Benchmark LLM','llm_'),('Benchmark embeddings','emb_')]:
                s=g[p+c];delta=s.diff().dropna();v=s.value_counts();prob=v/len(s)
                rows.append(dict(country=country,concept=c,method=method,n_levels=len(s),n_pairs=len(delta),zero_changes=int(delta.eq(0).sum()),zero_rate=delta.eq(0).mean(),
                    iid_tie_rate=(prob**2).sum(),permutation_tie_rate=(v*(v-1)).sum()/(len(s)*(len(s)-1)) if len(s)>1 else np.nan,
                    excess_over_iid=delta.eq(0).mean()-(prob**2).sum(),level_sd=s.std(ddof=0),change_sd=delta.std(ddof=0)))
    for name,frame in [('benchmark_tie_stickiness',pd.DataFrame(rows)),('benchmark_point_masses',pd.DataFrame(mass)),('benchmark_rating_distribution',pd.DataFrame(distribution))]:frame.to_csv(ROOT/'results'/f'{name}.csv',index=False)
    d['quote_prefix_verified']=[quote_present(q,t[:4500]) for q,t in zip(d.llm_quote,d.text)]
    d['quote_full_verified']=[quote_present(q,t) for q,t in zip(d.llm_quote,d.text)]
    d['quote_prefix_exact']=[bool(q) and q in t[:4500] for q,t in zip(d.llm_quote,d.text)]
    d[['doc_id','country','date','llm_quote','quote_prefix_exact','quote_prefix_verified','quote_full_verified',*['llm_'+c for c in CONCEPTS]]].to_csv(ROOT/'audit/benchmark_quote_audit.csv',index=False)
    qc=d.groupby('country').agg(n=('doc_id','size'),exact_prefix=('quote_prefix_exact','mean'),normalized_prefix=('quote_prefix_verified','mean'),normalized_full=('quote_full_verified','mean')).reset_index()
    qc.to_csv(ROOT/'results/benchmark_quotes_by_country.csv',index=False)
    qr=[]
    for c in CONCEPTS:
        for verified,g in d.groupby('quote_prefix_verified'):
            qr.append(dict(concept=c,shared_document_quote_verified=verified,n=len(g),mean_rating=g['llm_'+c].mean(),mass_at_4=g['llm_'+c].eq(4).mean(),overall_verification=d.quote_prefix_verified.mean()))
    pd.DataFrame(qr).to_csv(ROOT/'results/benchmark_quotes_by_concept.csv',index=False)
    duplicates=d.groupby('country').agg(n=('doc_id','size'),duplicate_documents=('duplicate_text','sum'),median_words=('word_count','median'))
    duplicates['duplicate_share']=duplicates.duplicate_documents/duplicates.n
    duplicates.reset_index().to_csv(ROOT/'results/duplicate_text_by_country.csv',index=False)
    trunc=pd.DataFrame([dict(method=m,cap_characters=cap,truncated_documents=int((d.characters>cap).sum()),share=(d.characters>cap).mean()) for m,cap in [('Benchmark direct LLM',4500),('Benchmark embeddings',3500)]]);trunc.to_csv(ROOT/'results/benchmark_truncation.csv',index=False)
    # Exact empirical marginal mapping preserves each country's whole grid, including ties.
    mapped=pd.DataFrame(index=d.index,columns=CONCEPTS,dtype=float)
    for c in CONCEPTS:
        for bank,g in d.groupby('country'):
            order=g.sort_values([c,'date','doc_id'],kind='stable').index
            mapped.loc[order,c]=np.sort(g['llm_'+c].to_numpy())
    methods={'lex_':d[CONCEPTS],'old_':d[['llm_'+c for c in CONCEPTS]].set_axis(CONCEPTS,axis=1),'emb_':d[['emb_'+c for c in CONCEPTS]].set_axis(CONCEPTS,axis=1),'grid_':mapped}
    model=d.copy()
    for pre,v in methods.items():model=add_method(model,v,pre)
    mat=match_leaders(model,list(methods));mat=mat[mat.union45]
    out=[]
    for c in CONCEPTS:
        # Lock all three methods and mapped series to identical observations.
        required=[side+p+c for side in ['z_','fed_zd_','ecb_zd_'] for p in methods]
        sample=mat.dropna(subset=required)
        for name,y,xf,xe in [('continuous','z_lex_','fed_zd_lex_','ecb_zd_lex_'),('grid_leader_only','z_lex_','fed_zd_grid_','ecb_zd_grid_'),('grid_both','z_grid_','fed_zd_grid_','ecb_zd_grid_'),('benchmark_llm','z_old_','fed_zd_old_','ecb_zd_old_'),('benchmark_embeddings','z_emb_','fed_zd_emb_','ecb_zd_emb_')]:
            rr,_=fit(sample,y+c,[xf+c,xe+c,'log_words'])
            for r in rr:
                if r['term']==xf+c:out.append(dict(concept=c,method=name,**r))
    out=bh(pd.DataFrame(out),['method'],family_size=7);out.to_csv(ROOT/'results/granularity_experiment.csv',index=False)
    gr=[]
    for c,g in out.groupby('concept'):
        vals=g.set_index('method').estimate;b=vals.get('continuous');ll=vals.get('benchmark_llm');mapped_b=vals.get('grid_both')
        gr.append(dict(concept=c,continuous=b,mapped=mapped_b,benchmark_llm=ll,original_gap=b-ll,remaining_gap=mapped_b-ll,
            gap_fraction_remaining=(mapped_b-ll)/(b-ll) if abs(b-ll)>1e-10 else np.nan))
    pd.DataFrame(gr).to_csv(ROOT/'results/granularity_gap_decomposition.csv',index=False)
    # An audit rule, not source verification: dates/location/president boilerplate only.
    def substantive_text(t):
        t=norm_quote(t)
        t=re.sub(r'the president of the ecb will comment.*?(?:today\.|$)',' ',t)
        t=re.sub(r'at today.s meeting[^:]*:',' ',t)
        t=re.sub(r'\b(?:monday|tuesday|wednesday|thursday|friday|january|february|march|april|may|june|july|august|september|october|november|december)\b',' ',t)
        return re.sub(r'\W+',' ',t).strip()
    pairs=read_jsonl(ROOT/'audit/leader_redlines.jsonl')
    for p in pairs:p['boilerplate_only_rule']=substantive_text(p['old_text'])==substantive_text(p['new_text'])
    pq=pd.DataFrame([{k:p[k] for k in ['pair_id','country','source','date','identical','word_similarity','boilerplate_only_rule']} for p in pairs])
    pq.to_csv(ROOT/'audit/leader_pair_flags.csv',index=False)
    pq.groupby(['country','source']).agg(n_pairs=('pair_id','size'),identical_rate=('identical','mean'),boilerplate_only_rate=('boilerplate_only_rule','mean'),median_word_similarity=('word_similarity','median')).reset_index().to_csv(ROOT/'results/leader_pair_quality.csv',index=False)
    model.drop(columns='text').to_csv(ROOT/'derived/diagnostic_scores.csv',index=False)
    print(pd.DataFrame(rows).query('country in ["US","ECB"]')[['country','concept','method','zero_rate']].to_string(index=False))
    print('Point masses:',mass);print('Quote prefix matches',d.quote_prefix_verified.sum(),'exact',d.quote_prefix_exact.sum(),'of',len(d));print('Granularity:',out[['concept','method','estimate','std_error']].to_string(index=False))
if __name__=='__main__':run()


### Analysis

In [ ]:
%%writefile code/analysis.py
"""Frozen estimands; run only after all requested annotations are available."""
from data import *
from estimation import *
import json

COVERAGE_THRESHOLD=1.0
HORIZONS=[7,14,21,30,45]
def save(df,name):df.to_csv(ROOT/'results'/f'{name}.csv',index=False);return df
def annotations():
    rows=[];structure=[]
    texts={r['doc_id']:r['text'] for r in read_jsonl(ROOT/'derived/corpus.jsonl')+read_jsonl(ROOT/'derived/ecb_intro_corpus.jsonl')}
    redlines={r['pair_id']:r for r in read_jsonl(ROOT/'audit/leader_redlines.jsonl')}
    for kind in ['level','prefix']:
        path=ROOT/'derived'/f'{kind}_annotations.jsonl'
        if not path.exists():continue
        for r in read_jsonl(path):
            for c,a in r['annotation']['concepts'].items():
                rows.append(dict(doc_id=r['doc_id'],country=r['country'],date=r['date'],kind=kind,draw=r['draw'],concept=c,
                    score=a['score'],coverage=a['coverage'],quote=a['quote'],quote_verified=r['evidence_verification'][c]['quote_verified'],
                    quote_exact=bool(a['quote']) and a['quote'] in (texts[r['doc_id']][:4500] if kind=='prefix' else texts[r['doc_id']]),cache_key=r['cache_key']))
            supplied=texts[r['doc_id']][:4500] if kind=='prefix' else texts[r['doc_id']]
            structured=dict(r['annotation']['structure'])
            structured['claims']=[dict(a,quote_exact=bool(a['quote']) and a['quote'] in supplied,quote_verified=quote_present(a['quote'],supplied)) for a in structured['claims']]
            structured['evidence_verification']={field:dict(quote_exact=bool(structured[field]) and structured[field] in supplied,quote_verified=quote_present(structured[field],supplied)) for field in ['rate_action_quote','balance_sheet_quote','guidance_quote','dissent_quote']}
            structure.append(dict(doc_id=r['doc_id'],country=r['country'],date=r['date'],kind=kind,draw=r['draw'],cache_key=r['cache_key'],**structured))
    scores=pd.DataFrame(rows);scores['valid']=scores.coverage.ge(COVERAGE_THRESHOLD)&scores.score.notna()&scores.quote_verified
    scores['usable_score']=scores.score.where(scores.valid)
    scores.to_csv(ROOT/'derived/continuous_scores_long.csv',index=False)
    with (ROOT/'derived/structured_claims.jsonl').open('w') as f:
        for r in structure:f.write(json.dumps(r,ensure_ascii=False)+'\n')
    pairs=[]
    for r in read_jsonl(ROOT/'derived/pair_annotations.jsonl'):
        for c,a in r['annotation']['concepts'].items():
            ev=r['evidence_verification'][c]
            valid=a['transition']=='comparable' and min(a['coverage_before'],a['coverage_after'])>=COVERAGE_THRESHOLD and ev['old_verified'] and ev['new_verified'] and bool(a['old_quote']) and bool(a['new_quote'])
            pairs.append(dict(pair_id=r['pair_id'],doc_id=r['doc_id'],country=r['country'],source=r['source'],date=r['date'],draw=r['draw'],concept=c,
                **a,old_quote_verified=ev['old_verified'],new_quote_verified=ev['new_verified'],valid=valid,usable_change=a['change'] if valid else np.nan,
                old_quote_exact=bool(a['old_quote']) and a['old_quote'] in redlines[r['pair_id']]['old_text'],new_quote_exact=bool(a['new_quote']) and a['new_quote'] in redlines[r['pair_id']]['new_text'],
                edit_type=r['annotation']['edit_type'],cache_key=r['cache_key']))
    pairs=pd.DataFrame(pairs);pairs.to_csv(ROOT/'derived/pair_scores_long.csv',index=False)
    return scores,pairs,structure
def build_scores(scores,pairs):
    raw=pd.read_json(ROOT/'derived/corpus.jsonl',lines=True);raw.date=pd.to_datetime(raw.date)
    raw=raw.sort_values(['country','date']).reset_index(drop=True)
    lex=pd.read_csv(ROOT/'derived/lexical_scores.csv').set_index('doc_id')
    d=add_method(raw,lex.loc[raw.doc_id,CONCEPTS].reset_index(drop=True),'lex_')
    full=scores[scores.kind=='level'];wide=full.pivot_table(index='doc_id',columns=['draw','concept'],values='usable_score',dropna=False)
    for draw in [1,2]:
        values=wide[draw].reindex(index=d.doc_id,columns=CONCEPTS).reset_index(drop=True)
        d=add_method(d,values,f'draw{draw}_')
    # Require both draws for the average; missing attention is not zero-valued sentiment.
    vals=pd.DataFrame({c:d[[f'draw1_{c}',f'draw2_{c}']].mean(axis=1).where(d[[f'draw1_{c}',f'draw2_{c}']].notna().all(axis=1)) for c in CONCEPTS})
    d=add_method(d,vals,'llm_')
    emb=pd.read_csv(ROOT/'derived/full_embedding_scores.csv');ew=emb[emb.version=='full'].set_index('doc_id')
    d=add_method(d,ew.reindex(d.doc_id)[CONCEPTS].reset_index(drop=True),'emb_')
    for c in CONCEPTS:
        for draw in [1,2]:
            p=pairs[(pairs.source=='mpsd')&(pairs.draw==draw)&(pairs.concept==c)].set_index('doc_id').usable_change
            d[f'pair{draw}_{c}']=d.doc_id.map(p)
        mean=d[[f'pair1_{c}',f'pair2_{c}']].mean(axis=1).where(d[[f'pair1_{c}',f'pair2_{c}']].notna().all(axis=1))
        d['zd_pair_'+c]=mean.groupby(d.country).transform(z)
        # Same scale for independent draw IV; preserve individual draw noise.
        for bank,g in d.groupby('country'):
            sd=np.sqrt((g[f'pair1_{c}'].var(ddof=0)+g[f'pair2_{c}'].var(ddof=0))/2)
            for draw in [1,2]:d.loc[g.index,f'zd_pair{draw}_{c}']=g[f'pair{draw}_{c}']/sd if pd.notna(sd) and sd>1e-12 else np.nan
    d.drop(columns='text').to_csv(ROOT/'derived/analysis_statement_scores.csv',index=False)
    return d
def main_models(d):
    prefixes=['lex_','llm_','emb_','draw1_','draw2_','pair_','pair1_','pair2_']
    m=match_leaders(d,prefixes)
    primary=m[m.days_since_fed.between(1,45)&m.days_until_fed.between(1,45)].copy()
    result=[];same=[];sign=[];horiz=[];sensitivity=[]
    specifications=[('TF-IDF','lex_','lex_'),('Continuous LLM','llm_','llm_'),('Full embeddings','emb_','emb_'),('Pairwise leader / lexical outcome','lex_','pair_')]
    for method,outcome,leader in specifications:
        for c in CONCEPTS:
            y='z_'+outcome+c;past='fed_zd_'+leader+c;future='future_zd_'+leader+c
            common=primary.dropna(subset=[y,past,future])
            for spec,terms,contrasts in [('preceding_only',[past,'log_words'],None),('joint',[past,future,'log_words'],{'past_minus_future':{past:1,future:-1}})]:
                rr,_=fit(common,y,terms,contrasts=contrasts)
                for r in rr:
                    if r['term']!='log_words':result.append(dict(method=method,concept=c,spec=spec,**r))
            for h in HORIZONS:
                sample=m[m.days_since_fed.between(1,h)&m.days_until_fed.between(1,45)].dropna(subset=[y,past,future])
                rr,_=fit(sample,y,[past,future,'log_words'],contrasts={'past_minus_future':{past:1,future:-1}})
                for r in rr:
                    if r['term']=='past_minus_future':horiz.append(dict(method=method,concept=c,horizon=h,**r))
    result=bh(pd.DataFrame(result),['method','spec','term'],family_size=7)
    # Terms contain concept names, so correct explicitly over the seven concepts for each leg.
    result['leg']=result.term.map(lambda t:'difference' if t=='past_minus_future' else ('future' if t.startswith('future') else 'past'))
    result=bh(result,['method','spec','leg'],family_size=7)
    result=bh(result,['spec','leg'],q='q_all_methods',family_size=28)
    save(result,'main_results');save(bh(pd.DataFrame(horiz),['method'],family_size=35),'horizon_joint_results')
    precision=result.copy()
    precision['ci90_low']=precision.estimate-stats.t.ppf(.95,precision.df_t)*precision.std_error
    precision['ci90_high']=precision.estimate+stats.t.ppf(.95,precision.df_t)*precision.std_error
    precision['equivalent_within_005']=precision.ci90_low.gt(-.05)&precision.ci90_high.lt(.05)
    precision['equivalent_within_010']=precision.ci90_low.gt(-.10)&precision.ci90_high.lt(.10)
    precision['approx_80pct_mde']=(stats.t.ppf(.975,precision.df_t)+stats.norm.ppf(.8))*precision.std_error
    save(precision,'precision_and_equivalence')
    # Leader-only comparisons: fixed lexical outcome, identical foreign sample and controls.
    for c in CONCEPTS:
        req=['z_lex_'+c]+['fed_zd_'+p+c for p in ['lex_','llm_','emb_','pair_']]
        sample=primary.dropna(subset=req)
        for pre in ['lex_','llm_','emb_','pair_']:
            rr,_=fit(sample,'z_lex_'+c,['fed_zd_'+pre+c,'log_words'])
            for r in rr:
                if r['term']!='log_words':same.append(dict(concept=c,leader_method=pre,**r))
        ss=[r for r in same if r['concept']==c]
        values={r['leader_method']:r['estimate'] for r in ss}
        sign.append(dict(concept=c,n=len(sample),lexical=values.get('lex_'),semantic_level=values.get('llm_'),embedding=values.get('emb_'),pairwise=values.get('pair_'),
            lexical_semantic_sign_flip=bool(values.get('lex_',np.nan)*values.get('llm_',np.nan)<0),lexical_pair_sign_flip=bool(values.get('lex_',np.nan)*values.get('pair_',np.nan)<0)))
    save(bh(pd.DataFrame(same),['leader_method'],family_size=7),'leader_only_common_sample');save(pd.DataFrame(sign),'sign_stability')
    common_results=[]
    for c in CONCEPTS:
        required=[]
        for method,outcome,leader in specifications:required += ['z_'+outcome+c,'fed_zd_'+leader+c,'future_zd_'+leader+c]
        common=primary.dropna(subset=list(dict.fromkeys(required)))
        for method,outcome,leader in specifications:
            past='fed_zd_'+leader+c;future='future_zd_'+leader+c
            rr,_=fit(common,'z_'+outcome+c,[past,future,'log_words'],contrasts={'past_minus_future':{past:1,future:-1}})
            common_results += [dict(method=method,concept=c,**r) for r in rr if r['term']=='past_minus_future']
    save(bh(pd.DataFrame(common_results),['method'],family_size=7),'method_common_sample_results')
    flags=pd.read_csv(ROOT/'audit/heading_date_candidates.csv');bad=set(flags.doc_id)
    base=primary.copy();base['month']=base.date.dt.to_period('M').astype(str)
    lag=d.sort_values(['country','date']).copy()
    for c in CONCEPTS:lag['ownlag_'+c]=lag.groupby('country')['z_lex_'+c].shift(1)
    base=base.merge(lag[['doc_id',*['ownlag_'+c for c in CONCEPTS]]],on='doc_id',validate='one_to_one')
    base['event_weight']=1/base.groupby('fed_date').doc_id.transform('size')
    for c in CONCEPTS:
        past='fed_zd_lex_'+c;future='future_zd_lex_'+c;y='z_lex_'+c
        variants=[('drop_duplicate_foreign',base[~base.duplicate_text],None,None),('drop_heading_candidates',base[~base.doc_id.isin(bad)],None,None),('drop_2020_2022',base[~base.year.between(2020,2022)],None,None),('own_previous_statement',base,'ownlag_'+c,None),('equal_events',base,None,'event_weight'),('country_month_clusters',base,None,None),('drop_KZ_PK',base[~base.country.isin(['KZ','PK'])],None,None)]
        for label,sample,extra,weight in variants:
            rr,_=fit(sample,y,[past,future,'log_words']+([extra] if extra else []),clusters=('country','month') if label=='country_month_clusters' else ('country','fed_date'),weight=weight,contrasts={'past_minus_future':{past:1,future:-1}})
            sensitivity += [dict(spec=label,concept=c,**r) for r in rr if r['term']=='past_minus_future']
    save(bh(pd.DataFrame(sensitivity),['spec'],family_size=7),'joint_sensitivity')
    m.drop(columns='text').to_csv(ROOT/'derived/analysis_matched_statements.csv',index=False)
    return m,primary
def reliability(scores,pairs,d):
    out=[];attention=[]
    for kind,data,value,key in [('level',scores[(scores.kind=='level')&scores.doc_id.isin(d.doc_id)],'usable_score','doc_id'),('pair',pairs[pairs.source=='mpsd'],'usable_change','pair_id')]:
        for (country,c),g in data.groupby(['country','concept']):
            p=g.pivot(index=key,columns='draw',values=value).reindex(columns=[1,2]).dropna()
            n=len(p);a=p[1];b=p[2];va=a.var(ddof=0);vb=b.var(ddof=0)
            concordance=2*a.cov(b,ddof=0)/(va+vb+(a.mean()-b.mean())**2) if n>1 and va+vb>0 else np.nan
            out.append(dict(kind=kind,country=country,concept=c,n_paired=n,correlation=a.corr(b) if n>2 and min(va,vb)>0 else np.nan,concordance=concordance,mae=(a-b).abs().mean(),mean_draw_difference=(a-b).mean(),tie_between_draws=(a==b).mean()))
        for c,g in data.groupby('concept'):
            p=g.pivot(index=key,columns='draw',values=value).reindex(columns=[1,2]).dropna();a=p[1];b=p[2]
            out.append(dict(kind=kind,country='ALL',concept=c,n_paired=len(p),correlation=a.corr(b),concordance=2*a.cov(b,ddof=0)/(a.var(ddof=0)+b.var(ddof=0)+(a.mean()-b.mean())**2),mae=(a-b).abs().mean(),mean_draw_difference=(a-b).mean(),tie_between_draws=(a==b).mean()))
    save(pd.DataFrame(out),'test_retest_reliability')
    full=scores[(scores.kind=='level')&scores.doc_id.isin(d.doc_id)].copy();full['year']=pd.to_datetime(full.date).dt.year
    att=full.groupby(['doc_id','country','year','concept']).agg(coverage=('coverage','mean'),covered=('valid','mean')).reset_index()
    country_year=att.groupby(['country','year','concept']).agg(coverage=('coverage','mean'),covered=('covered','mean'),n=('doc_id','size')).reset_index()
    save(country_year,'attention_country_year')
    annual=country_year.groupby(['year','concept']).agg(mean_coverage=('coverage','mean'),covered_share=('covered','mean'),countries=('country','nunique')).reset_index()
    save(annual,'attention_annual_equal_country')
    balanced=country_year[country_year.year.between(2005,2024)].groupby(['country','concept']).year.nunique()
    keep=set(balanced[balanced==20].index)
    b=country_year[[((r.country,r.concept) in keep and 2005<=r.year<=2024) for r in country_year.itertuples()]]
    save(b.groupby(['year','concept']).agg(mean_coverage=('coverage','mean'),covered_share=('covered','mean'),countries=('country','nunique')).reset_index(),'attention_balanced_2005_2024')
    qr=full.groupby(['country','concept']).agg(n=('doc_id','size'),nonempty_quotes=('quote',lambda s:s.ne('').sum()),exact_quotes=('quote_exact','sum'),verified_quotes=('quote_verified','sum'),covered=('valid','sum')).reset_index()
    qr['verification_given_quote']=qr.verified_quotes/qr.nonempty_quotes.replace(0,np.nan);save(qr,'new_evidence_verification')
    heaping=[]
    for c,g in full.groupby('concept'):
        valid=g.loc[g.valid,'score'];heaping.append(dict(concept=c,n_annotations=len(g),assessable=len(valid),absent_share=g.coverage.eq(0).mean(),
            mean_coverage=g.coverage.mean(),unique_scores=valid.nunique(),mass_at_50=valid.eq(50).mean(),integer_share=np.isclose(valid,valid.round()).mean(),multiple_of_five_share=np.isclose(valid/5,(valid/5).round()).mean()))
    save(pd.DataFrame(heaping),'continuous_score_heaping')
    tie=[]
    for bank in ['US','ECB']:
        g=d[d.country==bank]
        for c in CONCEPTS:
            for label,col,differenced in [('TF-IDF','lex_'+c,True),('Full embedding','emb_'+c,True),('Continuous LLM','llm_'+c,True),('Direct pair draw 1','pair1_'+c,False),('Direct pair draw 2','pair2_'+c,False)]:
                delta=g[col].diff() if differenced else g[col]
                tie.append(dict(country=bank,concept=c,method=label,total_pairs=max(0,len(g)-1),assessable_pairs=delta.notna().sum(),zero_changes=delta.eq(0).sum(),zero_rate_assessable=delta.dropna().eq(0).mean(),missing_rate=delta.iloc[1:].isna().mean()))
    save(pd.DataFrame(tie),'new_leader_zero_rates')
    conv=[]
    old=pd.read_csv(ROOT/'derived/diagnostic_scores.csv',parse_dates=['date']).set_index('doc_id')
    for bank in ['US','ECB']:
        g=d[d.country==bank]
        for c in CONCEPTS:
            pair=g[['pair1_'+c,'pair2_'+c]].mean(axis=1).where(g[['pair1_'+c,'pair2_'+c]].notna().all(axis=1))
            for label,col in [('TF-IDF','lex_'),('Full embedding','emb_'),('Full continuous','llm_')]:
                diff=g[col+c].diff();valid=pair.notna()&diff.notna();conv.append(dict(country=bank,concept=c,method=label,n=valid.sum(),correlation=pair[valid].corr(diff[valid])))
            for label,col in [('Cached coarse','old_'),('Cached embeddings','emb_')]:
                diff=old.reindex(g.doc_id)[col+c].diff();diff.index=g.index;valid=pair.notna()&diff.notna();conv.append(dict(country=bank,concept=c,method=label,n=valid.sum(),correlation=pair[valid].corr(diff[valid])))
    save(pd.DataFrame(conv),'pair_convergent_validity')
    common_ties=[]
    for bank in ['US','ECB']:
        g=d[d.country==bank]
        for c in CONCEPTS:
            a=g['pair1_'+c];b=g['pair2_'+c]
            shared=a.notna()&b.notna()
            for label,s in [('Pair draw 1',a),('Pair draw 2',b),('TF-IDF',g['lex_'+c].diff()),('Continuous LLM',g['llm_'+c].diff()),('Full embedding',g['emb_'+c].diff())]:
                valid=shared&s.notna();common_ties.append(dict(country=bank,concept=c,method=label,n_shared_pair_assessable=int(shared.sum()),n_valid=int(valid.sum()),zero_rate=s[valid].eq(0).mean()))
    save(pd.DataFrame(common_ties),'common_assessable_zero_rates')
def phrase_claim_tests(d,m,structure):
    novel=json.loads((ROOT/'audit/novel_phrase_candidates.json').read_text());meaningful={}
    adjudicated=pd.read_csv(ROOT/'derived/meaningful_novel_phrases.csv')
    for doc,g in adjudicated.groupby('doc_id'):
        if set(g.phrase)!=set(novel[doc]):raise RuntimeError('Incomplete novel-phrase adjudication for '+doc)
        meaningful[doc]=set(g.loc[g.meaningful,'phrase'])
    fed=d[d.country=='US'].sort_values('date');date_id=dict(zip(fed.date,fed.doc_id));grams={r.doc_id:content_ngrams(r.text) for r in d.itertuples() if r.country not in ['US','ECB']}
    records=[]
    for r in m.itertuples():
        if r.fed_date not in date_id or r.future_date not in date_id:continue
        pid=date_id[r.fed_date];fid=date_id[r.future_date]
        names_fed=bool(re.search(r'\bfederal\s+reserve\b|\bfomc\b|\bthe\s+fed\b',r.text,re.I))
        for variant,bankgrams in [('all_novel',novel),('meaningful_novel',meaningful)]:
            past=set(bankgrams.get(pid,[]));future=set(bankgrams.get(fid,[]));own=grams[r.doc_id]
            matched_p=sorted(past&own);matched_f=sorted(future&own)
            records.append(dict(doc_id=r.doc_id,country=r.country,year=r.year,date=r.date,fed_date=r.fed_date,future_date=r.future_date,days_since_fed=r.days_since_fed,days_until_fed=r.days_until_fed,
                names_fed=names_fed,english_original=r.english_original,official_english=int(r.country in SOURCE_OFFICIAL_ENGLISH),variant=variant,n_past_novel=len(past),n_future_novel=len(future),past_share=len(matched_p)/len(past) if past else np.nan,future_share=len(matched_f)/len(future) if future else np.nan,
                past_matches=json.dumps(matched_p),future_matches=json.dumps(matched_f)))
    records=pd.DataFrame(records);records['difference']=100*(records.past_share-records.future_share)
    records.to_csv(ROOT/'audit/phrase_matches.csv',index=False)
    out=[]
    for variant,g in records.groupby('variant'):
        for split,mask in [('all',pd.Series(True,index=g.index)),('names_Fed',g.names_fed),('does_not_name_Fed',~g.names_fed),('Official_English',g.official_english.eq(1)),('Database_translated',g.official_english.eq(0))]:
            for h in HORIZONS:
                for design in ['past_window','symmetric_window']:
                    s=g[mask&g.days_since_fed.between(1,h)&g.days_until_fed.between(1,h if design=='symmetric_window' else 45)].copy()
                    s=s.dropna(subset=['past_share','future_share'])
                    s['intercept']=1.0
                    rr,_=fit(s,'difference',['intercept'],fe=(),clusters=('country','fed_date'))
                    # 'intercept' plus automatic constant would be collinear; fit() handles named intercept below.
                    if rr:
                        r=rr[0];out.append(dict(variant=variant,split=split,horizon=h,design=design,past_mean_pp=100*s.past_share.mean(),future_mean_pp=100*s.future_share.mean(),**r))
                    else:out.append(dict(variant=variant,split=split,horizon=h,design=design,n=len(s),estimate=np.nan,std_error=np.nan,p_value=np.nan,status='insufficient_support'))
    out=bh(pd.DataFrame(out),['variant','design'],family_size=25);save(out,'phrase_adoption')
    if structure is None:return
    claim_tests(d,m,structure)
def claim_tests(d,m,structure):
    """Claim-type alignment, separately by repeated draw and evidence screen.

    A new claim type is absent from the immediately preceding Fed statement, unlike
    literal novelty, which uses all earlier observed Fed statements. Neither records
    whether a foreign bank learned the proposition from the Fed.
    """
    out=[];audit=[]
    for draw in [1,2]:
        for evidence in ['verified_quotes','all_extracted']:
            cr,co=claim_test_variant(d,m,structure,draw,evidence)
            audit.append(cr);out.extend(co)
    pd.concat(audit,ignore_index=True).to_csv(ROOT/'audit/claim_alignment.csv',index=False)
    save(bh(pd.DataFrame(out),['draw','evidence'],family_size=20),'claim_alignment_horizons')
def claim_test_variant(d,m,structure,draw,evidence):
    fed=d[d.country=='US'].sort_values('date');date_id=dict(zip(fed.date,fed.doc_id))
    structs={r['doc_id']:r for r in structure if r['kind']=='level' and r['draw']==draw}
    claimsets={}
    for doc,r in structs.items():
        for scope in ['domestic','US_reporting']:
            # For a Fed document, "domestic" and explicitly "US" refer to the
            # same economy. Preserve the distinction for foreign statements.
            def in_scope(c):
                if scope=='domestic':return c['geography']=='domestic' or (r['country']=='US' and c['geography']=='US')
                return c['geography']=='US'
            claimsets[doc,scope]={(c['concept'],c['horizon'],c['direction']) for c in r['claims'] if in_scope(c) and (evidence=='all_extracted' or c['quote_verified'])}
    leader_new={};seen=set()
    for index,r in enumerate(fed.itertuples()):
        claims=claimsets.get((r.doc_id,'domestic'),set());leader_new[r.doc_id]=claims-seen if index else set();seen=claims
    cr=[]
    for r in m.itertuples():
        if r.fed_date not in date_id or r.future_date not in date_id:continue
        for scope in ['domestic','US_reporting']:
            for concept in ['inflation','activity']:
                past={x for x in leader_new.get(date_id[r.fed_date],set()) if x[0]==concept};future={x for x in leader_new.get(date_id[r.future_date],set()) if x[0]==concept};own=claimsets.get((r.doc_id,scope),set())
                cr.append(dict(doc_id=r.doc_id,country=r.country,fed_date=r.fed_date,days_since_fed=r.days_since_fed,days_until_fed=r.days_until_fed,scope=scope,concept=concept,draw=draw,evidence=evidence,n_new_past=len(past),n_new_future=len(future),past_matched=len(own&past),future_matched=len(own&future),difference=100*(len(own&past)/len(past)-len(own&future)/len(future)) if past and future else np.nan))
    cr=pd.DataFrame(cr);co=[]
    for (scope,c),g in cr.groupby(['scope','concept']):
        for h in HORIZONS:
            s=g[g.days_since_fed.between(1,h)&g.days_until_fed.between(1,45)].copy();s['intercept']=1.
            rr,_=fit(s,'difference',['intercept'],fe=())
            co += [dict(scope=scope,concept=c,horizon=h,draw=draw,evidence=evidence,**r) for r in rr]
    return cr,co
def structured_diagnostics(structure,d):
    from sklearn.metrics import cohen_kappa_score
    rows=[r for r in structure if r['kind']=='level' and r['doc_id'] in set(d.doc_id)]
    frame=pd.DataFrame(rows);frame['year']=pd.to_datetime(frame.date).dt.year
    fields=['rate_action','balance_sheet_action','guidance','dissent'];agreement=[]
    for field in fields:
        wide=frame.pivot(index='doc_id',columns='draw',values=field).reindex(columns=[1,2]).dropna()
        informative=~wide[1].isin(['not_stated','none','not_mentioned','unclear'])&~wide[2].isin(['not_stated','none','not_mentioned','unclear'])
        for label,s in [('all_documents',wide),('both_informative',wide[informative])]:
            agreement.append(dict(field=field,sample=label,n=len(s),exact_agreement=s[1].eq(s[2]).mean(),kappa=cohen_kappa_score(s[1],s[2]) if len(s) else np.nan))
    save(pd.DataFrame(agreement),'structured_annotation_reliability')
    annual=[]
    first=frame[frame.draw==1]
    for field in fields:
        for (country,year),g in first.groupby(['country','year']):
            for value,n in g[field].value_counts().items():annual.append(dict(country=country,year=year,field=field,value=value,n=n,share=n/len(g),documents=len(g)))
    save(pd.DataFrame(annual),'structured_content_country_year')
    claims=[]
    for r in first.to_dict('records'):
        for c in r['claims']:claims.append(dict(doc_id=r['doc_id'],country=r['country'],year=r['year'],**c))
    claims=pd.DataFrame(claims);claims.to_csv(ROOT/'derived/claims_draw1_long.csv',index=False)
    save(claims.groupby(['concept','geography','horizon','direction']).agg(claims=('doc_id','size'),documents=('doc_id','nunique')).reset_index(),'structured_claim_distribution')
    save(claims.groupby(['country','concept']).agg(claims=('doc_id','size'),documents=('doc_id','nunique'),exact_quotes=('quote_exact','sum'),verified_quotes=('quote_verified','sum')).reset_index(),'structured_claim_evidence')
    evidence=[]
    for r in rows:
        for field,a in r['evidence_verification'].items():evidence.append(dict(doc_id=r['doc_id'],country=r['country'],draw=r['draw'],field=field,nonempty=bool(r[field]),**a))
    ev=pd.DataFrame(evidence)
    save(ev.groupby(['country','field']).agg(annotations=('doc_id','size'),nonempty=('nonempty','sum'),exact_quotes=('quote_exact','sum'),verified_quotes=('quote_verified','sum')).reset_index(),'structured_policy_evidence')
    vals=[]
    lookup={(r['doc_id'],r['draw']):{(c['concept'],c['geography'],c['horizon'],c['direction']) for c in r['claims']} for r in rows}
    for doc in d.doc_id:
        a=lookup.get((doc,1),set());b=lookup.get((doc,2),set());vals.append(dict(doc_id=doc,draw1_claim_types=len(a),draw2_claim_types=len(b),claim_type_jaccard=len(a&b)/len(a|b) if a|b else np.nan,both_empty=not(a|b)))
    save(pd.DataFrame(vals),'claim_type_repeatability')
def run():
    scores,pairs,structure=annotations()
    raw=pd.read_json(ROOT/'derived/corpus.jsonl',lines=True)
    expected_docs=read_jsonl(ROOT/'derived/corpus.jsonl')+read_jsonl(ROOT/'derived/ecb_intro_corpus.jsonl')
    expected={r['doc_id'] for r in expected_docs}
    for draw in [1,2]:
        actual=set(scores[(scores.kind=='level')&(scores.draw==draw)].doc_id)
        missing=expected-actual
        if missing:raise RuntimeError(f'Full corpus draw {draw} incomplete: {len(missing)} documents. No revised baseline exported.')
    expected_prefix={r['doc_id'] for r in expected_docs if len(r['text'])>4500}
    missing_prefix=expected_prefix-set(scores.loc[scores.kind=='prefix','doc_id'])
    if missing_prefix:raise RuntimeError(f'Prefix robustness incomplete: {len(missing_prefix)} documents. No revised baseline exported.')
    expected_pairs={line['pair_id'] for line in read_jsonl(ROOT/'audit/leader_redlines.jsonl')}
    for draw in [1,2]:
        missing=expected_pairs-set(pairs.loc[pairs.draw==draw,'pair_id'])
        if missing:raise RuntimeError(f'Pair draw {draw} incomplete: {len(missing)} pairs.')
    d=build_scores(scores,pairs);m,primary=main_models(d);reliability(scores,pairs,d);phrase_claim_tests(d,m,structure);structured_diagnostics(structure,d)
    print('Main analysis complete',len(d),'statements',len(primary),'paired 45-day statements',flush=True)
if __name__=='__main__':run()


### Measurement checks

In [ ]:
%%writefile code/measurement_checks.py
"""Repeated-draw IV, coverage/prefix sensitivity and explicit ECB source repair."""
from data import *
from estimation import *
from analysis import save,COVERAGE_THRESHOLD
from scipy import stats,linalg

def ar_confidence(a,b,va,vb,cab,critical):
    A=b*b-critical**2*vb;B=-2*a*b+2*critical**2*cab;C=a*a-critical**2*va
    if abs(A)<1e-14:
        if abs(B)<1e-14:return 'all real numbers' if C<=0 else 'empty',np.nan,np.nan,False
        cut=-C/B;return ('(-inf, %.6g]'%cut if B>0 else '[%.6g, inf)'%cut),np.nan,np.nan,False
    disc=B*B-4*A*C
    if disc<0:return ('all real numbers' if A<0 else 'empty'),np.nan,np.nan,False
    lo,hi=sorted([(-B-np.sqrt(disc))/(2*A),(-B+np.sqrt(disc))/(2*A)])
    if A>0:return f'[{lo:.6g}, {hi:.6g}]',lo,hi,True
    return f'(-inf, {lo:.6g}] union [{hi:.6g}, inf)',np.nan,np.nan,False
def iv_fit(d,y,x,instrument):
    needed=[y,x,instrument,'log_words','country','year','fed_date'];d=d[needed].replace([np.inf,-np.inf],np.nan).dropna().copy()
    if len(d)<80 or d[instrument].std()<1e-12 or d[x].std()<1e-12:return dict(n=len(d),status='unidentified_or_insufficient_support')
    if min(d.country.nunique(),d.fed_date.nunique())<2:return dict(n=len(d),status='insufficient_clusters')
    reduced,m_y=fit(d,y,[instrument,'log_words']);first,m_x=fit(d,x,[instrument,'log_words'])
    if any(r['status']!='identified' for r in first):return dict(n=len(d),status='unidentified_first_stage_design')
    # Reuse the exact first-stage design; x replaces the instrument in the structural design.
    Z=m_x['X'].to_numpy();X=Z.copy();j=m_x['X'].columns.get_loc(instrument);X[:,j]=d[x].to_numpy()
    proj=Z@linalg.lstsq(Z,X,lapack_driver='gelsy')[0];bread=linalg.pinvh(proj.T@proj)
    beta=bread@proj.T@d[y].to_numpy();res=d[y].to_numpy()-X@beta;scores=proj*res[:,None]
    cov=np.zeros_like(bread);counts=[]
    for cols,sgn in [(['country'],1),(['fed_date'],1),(['country','fed_date'],-1)]:
        meat,g=cluster_meat(scores,pd.MultiIndex.from_frame(d[cols]));counts.append(g)
        cov+=sgn*g/(g-1)*(len(d)-1)/(len(d)-X.shape[1])*bread@meat@bread
    if not (np.isfinite(beta).all() and np.isfinite(cov).all()):raise ArithmeticError('Non-finite IV calculation; no result accepted')
    se=np.sqrt(cov[j,j]) if cov[j,j]>0 else np.nan;df=min(counts[:2])-1;crit=stats.t.ppf(.975,df)
    a=m_y['beta'][j];b=m_x['beta'][j];va=m_y['cov'][j,j];vb=m_x['cov'][j,j]
    d['sum_y_x']=d[y]+d[x];_,m_s=fit(d,'sum_y_x',[instrument,'log_words']);cab=(m_s['cov'][j,j]-va-vb)/2
    ar,low,high,bounded=ar_confidence(a,b,va,vb,cab,crit)
    first_f=b*b/vb if vb>0 else np.nan
    return dict(n=len(d),n_countries=counts[0],n_events=counts[1],estimate=beta[j],std_error=se,p_value=2*stats.t.sf(abs(beta[j]/se),df) if pd.notna(se) else np.nan,
        ci_low=beta[j]-crit*se,ci_high=beta[j]+crit*se,first_stage_coefficient=b,first_stage_F_type=first_f,first_stage_df=df,ar_95_set=ar,ar_lower=low,ar_upper=high,ar_bounded=bounded,
        ar_excludes_abs_010=bool(bounded and low>-.10 and high<.10),status='estimated_assumption_dependent')
def repeated_draw_iv(d,m):
    out=[]
    sample=m[m.days_since_fed.between(1,45)&m.days_until_fed.between(1,45)]
    for c in CONCEPTS:
        for draw,instrument in [(1,2),(2,1)]:
            out.append(dict(concept=c,endogenous_draw=draw,instrument_draw=instrument,**iv_fit(sample,'z_lex_'+c,f'fed_zd_pair{draw}_'+c,f'fed_zd_pair{instrument}_'+c)))
    save(bh(pd.DataFrame(out),['endogenous_draw'],family_size=7),'repeated_draw_iv')
def prefixes_and_coverage(d,scores):
    out=[];level=scores[scores.kind=='level'];pfx=scores[scores.kind=='prefix'];embed=pd.read_csv(ROOT/'derived/full_embedding_scores.csv')
    for c in CONCEPTS:
        for version in ['full_draw1','prefix_draw1']:
            s=level[(level.draw==1)&(level.concept==c)].set_index('doc_id').usable_score
            if version=='prefix_draw1':
                pre=pfx[pfx.concept==c].set_index('doc_id').usable_score;s=s.copy();s.loc[pre.index]=pre
            vals=d.doc_id.map(s)
            out.append(pd.DataFrame({'doc_id':d.doc_id,'concept':c,'version':version,'score':vals}))
    flat=pd.concat(out,ignore_index=True);save(flat,'full_prefix_score_comparison')
    models=[];correlations=[]
    for version,kind in [('full_draw1','llm'),('prefix_draw1','llm'),('full','emb'),('prefix','emb')]:
        if kind=='llm':v=flat[flat.version==version].pivot(index='doc_id',columns='concept',values='score').reindex(d.doc_id)[CONCEPTS].reset_index(drop=True)
        else:v=embed[embed.version==version].set_index('doc_id').reindex(d.doc_id)[CONCEPTS].reset_index(drop=True)
        model=add_method(d,v,'check_');mat=match_leaders(model,['check_']);mat=mat[mat.days_since_fed.between(1,45)&mat.days_until_fed.between(1,45)]
        for c in CONCEPTS:
            # Every version is reestimated on the same rows for that method/concept below.
            models.append((kind,version,c,mat))
    rows=[]
    for kind in ['llm','emb']:
        for c in CONCEPTS:
            ms=[x for x in models if x[0]==kind and x[2]==c];req=['z_check_'+c,'fed_zd_check_'+c,'future_zd_check_'+c]
            common=set.intersection(*[set(x[3].dropna(subset=req).doc_id) for x in ms])
            for _,version,_,mat in ms:
                sample=mat[mat.doc_id.isin(common)];rr,_=fit(sample,req[0],[req[1],req[2],'log_words'],contrasts={'past_minus_future':{req[1]:1,req[2]:-1}})
                rows += [dict(method=kind,version=version,concept=c,**r) for r in rr if r['term']=='past_minus_future']
    save(bh(pd.DataFrame(rows),['method','version'],family_size=7),'prefix_common_sample_results')
    cover=[]
    for threshold in [1,5,10]:
        s=level.copy();s['value']=s.score.where(s.quote_verified&s.coverage.ge(threshold));v=s.pivot_table(index='doc_id',columns=['draw','concept'],values='value')
        val=pd.DataFrame({c:v[[(1,c),(2,c)]].mean(axis=1).where(v[[(1,c),(2,c)]].notna().all(axis=1)) for c in CONCEPTS}).reindex(d.doc_id).reset_index(drop=True)
        model=add_method(d,val,'check_');mat=match_leaders(model,['check_']);mat=mat[mat.days_since_fed.between(1,45)&mat.days_until_fed.between(1,45)]
        for c in CONCEPTS:
            past='fed_zd_check_'+c;future='future_zd_check_'+c;rr,_=fit(mat,'z_check_'+c,[past,future,'log_words'],contrasts={'past_minus_future':{past:1,future:-1}})
            cover += [dict(coverage_threshold=threshold,concept=c,**r) for r in rr if r['term']=='past_minus_future']
    save(bh(pd.DataFrame(cover),['coverage_threshold'],family_size=7),'coverage_threshold_results')
def ecb_and_a4(d,m,pairs):
    v,w=fit_lexical(json.loads((ROOT/'data/exemplars.json').read_text()));intro=load_external_text('ecb_intro','ECB');s=lexical(intro.text,v,w);s.index=intro.index
    intro=add_method(intro,s[CONCEPTS],'lex_')
    intro['intro_previous_date']=intro.date.shift(1)
    raw_ecb=d[d.country=='ECB'].sort_values('date');raw_previous=pd.Series(raw_ecb.date.shift(1).to_numpy(),index=raw_ecb.date)
    # Only match to full introductions that share the MPSD event date; avoid bridging source changes.
    repaired=match_leaders(d,['lex_'],ecb=intro)
    flags=pd.read_csv(ROOT/'audit/leader_pair_flags.csv',parse_dates=['date']);oldflags=flags[(flags.country=='ECB')&(flags.source=='mpsd')].set_index('date').boilerplate_only_rule
    repaired['old_ecb_boilerplate']=repaired.ecb_date.map(oldflags)
    start=max(intro.date.min(),d[d.country=='ECB'].date.min());end=intro.date.max();rows=[]
    for c in CONCEPTS:
        # Fix the exact leader event date and foreign sample for raw versus introduction comparison.
        original=m[m.union45&m.ecb_date.between(start,end)].copy();original=original[original.ecb_date.isin(intro.date)]
        full=original[['doc_id','ecb_date']].merge(intro[['date','intro_previous_date','zd_lex_'+c]].rename(columns={'date':'ecb_date','zd_lex_'+c:'full_ecb'}),on='ecb_date',validate='many_to_one')
        original=original.merge(full[['doc_id','full_ecb','intro_previous_date']],on='doc_id',validate='one_to_one');original['old_ecb_boilerplate']=original.ecb_date.map(oldflags).fillna(False).astype(bool)
        original['same_previous_date']=original.ecb_date.map(raw_previous).eq(original.intro_previous_date)
        common=original.dropna(subset=['z_lex_'+c,'fed_zd_lex_'+c,'ecb_zd_lex_'+c,'full_ecb'])
        for label,sample,ec in [('raw_same_dates',common,'ecb_zd_lex_'+c),('full_intro_same_dates',common,'full_ecb'),('raw_excluding_boilerplate',common[~common.old_ecb_boilerplate],'ecb_zd_lex_'+c),('full_intro_excluding_boilerplate',common[~common.old_ecb_boilerplate],'full_ecb'),('raw_same_predecessor',common[common.same_previous_date],'ecb_zd_lex_'+c),('full_intro_same_predecessor',common[common.same_previous_date],'full_ecb')]:
            rr,_=fit(sample,'z_lex_'+c,['fed_zd_lex_'+c,ec,'log_words'],clusters=('country','fed_date','ecb_date'))
            rows += [dict(spec=label,concept=c,leader='Fed' if r['term'].startswith('fed_') else 'ECB',n_ecb_events=sample.ecb_date.nunique(),**r) for r in rr if r['term']!='log_words']
    repaired_results=bh(pd.DataFrame(rows),['spec','leader'],family_size=7)
    repaired_results=bh(repaired_results,['leader'],q='q_all_source_specs',family_size=42)
    save(repaired_results,'ecb_source_repair_same_dates')
    # A4 exactly preserves the original union, original foreign lexical score and ECB nuisance control.
    old=pd.read_csv(ROOT/'derived/diagnostic_scores.csv',parse_dates=['date'])
    old=old.merge(d[['doc_id',*['zd_pair_'+c for c in CONCEPTS]]],on='doc_id',validate='one_to_one')
    matched=match_leaders(old,['lex_','old_','emb_','pair_']);matched=matched[matched.union45];rows=[]
    for c in CONCEPTS:
        req=['z_lex_'+c,'ecb_zd_lex_'+c,*['fed_zd_'+p+c for p in ['lex_','old_','emb_','pair_']]];common=matched.dropna(subset=req)
        for pre in ['lex_','old_','emb_','pair_']:
            rr,_=fit(common,'z_lex_'+c,['fed_zd_'+pre+c,'ecb_zd_lex_'+c,'log_words']);rows += [dict(leader_method=pre,concept=c,**r) for r in rr if r['term'].startswith('fed_')]
    save(bh(pd.DataFrame(rows),['leader_method'],family_size=7),'union_leader_comparison')
def run():
    d=pd.read_csv(ROOT/'derived/analysis_statement_scores.csv',parse_dates=['date']);m=pd.read_csv(ROOT/'derived/analysis_matched_statements.csv',parse_dates=['date','fed_date','ecb_date','future_date']);scores=pd.read_csv(ROOT/'derived/continuous_scores_long.csv');pairs=pd.read_csv(ROOT/'derived/pair_scores_long.csv')
    repeated_draw_iv(d,m);prefixes_and_coverage(d,scores);ecb_and_a4(d,m,pairs)
    print('IV, coverage, prefix and ECB checks complete',flush=True)
if __name__=='__main__':run()


### Supplementary

In [ ]:
%%writefile code/supplementary.py
"""Re-estimate all retained annual, market, macro and sensitivity exhibits."""
from data import *
from estimation import *
from analysis import save
from scipy import stats
def annual(raw,lex):
    d=raw.merge(lex[['doc_id',*CONCEPTS]],on='doc_id',validate='one_to_one')
    cy=d.groupby(['country','year']).agg(**{c:(c,'mean') for c in CONCEPTS},iso3=('iso3','first'),advanced=('advanced','first'),english_original=('english_original','first'),inflation_targeter=('inflation_targeter','first'),statements=('doc_id','size')).reset_index()
    for c in CONCEPTS:
        cy['z_'+c]=cy.groupby('country')[c].transform(z)
        cy['lag_'+c]=cy.groupby('country')['z_'+c].shift(1).where(cy.groupby('country').year.diff().eq(1))
    rows=[];balanced=[]
    groups={'All':cy,'Excl. US/ECB':cy[~cy.country.isin(['US','ECB'])],'Advanced':cy[cy.advanced.eq(1)],'Other':cy[cy.advanced.eq(0)],'English':cy[cy.english_original.eq(1)]}
    for label,g in groups.items():
        for c in CONCEPTS:
            demean=g['z_'+c]-g.groupby('country')['z_'+c].transform('mean');yr=demean.groupby(g.year).transform('mean')
            share=100*float(yr@yr)/float(demean@demean)
            rows.append(dict(sample=label,concept=c,projection_share=share,n=len(g),countries=g.country.nunique()))
    subset=cy[cy.year.between(2005,2024)];complete=subset.groupby('country').year.nunique().eq(20)
    for label,g in [('All',cy),('Balanced',subset[subset.country.isin(complete[complete].index)])]:
        for c in CONCEPTS:
            g=g.copy();g['value']=g.groupby('country')[c].transform(z)
            r0,f0=fit(g,'value',[],fe=('country',),clusters=());r1,f1=fit(g,'value',[],fe=('country','year'),clusters=())
            demean=g.value-g.groupby('country').value.transform('mean');yr=demean.groupby(g.year).transform('mean')
            balanced.append(dict(sample=label,concept=c,projection_share=100*(yr@yr)/(demean@demean),joint_fe_partial_r2=100*(1-f1['resid']@f1['resid']/(f0['resid']@f0['resid'])),n=len(g),countries=g.country.nunique()))
    save(pd.DataFrame(rows),'annual_comovement');save(pd.DataFrame(balanced),'annual_balanced')
    follower=cy[~cy.country.isin(['US','ECB'])].copy()
    for bank,p in [('US','fed'),('ECB','ecb')]:
        leader=cy[cy.country==bank][['year',*['z_'+c for c in CONCEPTS]]].copy()
        for c in CONCEPTS:leader[p+'_lag_'+c]=leader['z_'+c].shift(1).where(leader.year.diff().eq(1))
        follower=follower.merge(leader[['year',*[p+'_lag_'+c for c in CONCEPTS]]],on='year',how='left',validate='many_to_one')
    out=[]
    for c in CONCEPTS:
        rr,_=fit(follower,'z_'+c,['lag_'+c,'fed_lag_'+c,'ecb_lag_'+c],fe=('country',),clusters=('country','year'))
        out += [dict(concept=c,**r) for r in rr]
    save(pd.DataFrame(out),'annual_leader_follower');cy.to_csv(ROOT/'derived/annual_panel.csv',index=False)
    return cy
def markets(v,w):
    from statsmodels.regression.linear_model import OLS
    def sentence_mean(t):
        sentences=[s.strip() for s in re.split(r'(?<=[.!?])\s+(?=[A-Z\"\'])',t) if s.strip()]
        return lexical(sentences or [t],v,w).mean().to_dict()
    minutes=load_external_text('fed_minutes','US');intro=load_external_text('ecb_intro','ECB');rows=[];samples=[]
    for source,d,workbook,sheet,outcomes,control in [('minutes',minutes,'USMPD.xlsx','Minutes',['UST2Y','UST10Y','SP500','TIPS5Y'],'MP2'),('ecb',intro,'Dataset_EA-MPD.xlsx','Press Conference Window',['OIS_2Y','DE10Y','STOXX50','EURUSD'],'OIS_1Y_pr')]:
        scores=pd.DataFrame([sentence_mean(t) for t in d.text]);scores.index=d.index;d=pd.concat([d,scores],axis=1)
        for c in HF_CONCEPTS:d['delta_'+c]=d[c].diff()
        market=pd.read_excel(ROOT/'data/external'/workbook,sheet_name=sheet);market.columns=[str(c).strip() for c in market.columns]
        date_col='Date' if source=='minutes' else 'date';market['date']=pd.to_datetime(market[date_col]).dt.normalize()
        for c in outcomes+([control] if source=='minutes' else []):market[c]=pd.to_numeric(market[c],errors='coerce')
        if source=='ecb':
            pr=pd.read_excel(ROOT/'data/external'/workbook,sheet_name='Press Release Window');pr.columns=[str(c).strip() for c in pr.columns];pr['date']=pd.to_datetime(pr.date).dt.normalize();market=market.merge(pr[['date','OIS_1Y']].rename(columns={'OIS_1Y':'OIS_1Y_pr'}),on='date',how='left',validate='one_to_one')
        sample=d.merge(market[['date',*outcomes,control]],on='date',how='inner',validate='one_to_one')
        for c in HF_CONCEPTS:sample['zd_'+c]=z(sample['delta_'+c])
        if source=='minutes':sample[control]*=100
        for outcome in outcomes:
            sample['outcome']=sample[outcome]*(100 if source=='minutes' and outcome.startswith(('UST','TIPS')) else 1)
            rr,_=fit(sample,'outcome',[*['zd_'+c for c in HF_CONCEPTS],control],fe=(),clusters=())
            rows += [dict(source=source,outcome=outcome,**r) for r in rr]
        sample.drop(columns='text').to_csv(ROOT/'derived'/f'{source}_market_sample.csv',index=False)
    out=pd.DataFrame(rows);mask=out.term.str.startswith('zd_');out.loc[mask,'q_value']=bh(out[mask],[],family_size=48).q_value
    save(out,'market_results')
def macro(cy):
    wb=pd.read_csv(ROOT/'data/external/world_bank_macro_raw.csv');wb.year=wb.year.astype(int);wb=wb.sort_values(['iso3','year'])
    gap=wb.groupby('iso3').year.diff().eq(1);leadgap=wb.groupby('iso3').year.shift(-1).sub(wb.year).eq(1)
    wb['inflation_lead1']=wb.groupby('iso3').inflation.shift(-1).where(leadgap)
    wb['inflation_lag1']=wb.groupby('iso3').inflation.shift(1).where(gap)
    wb['gdp_growth_lag1']=wb.groupby('iso3').gdp_growth.shift(1).where(gap)
    wb['fx_depreciation']=np.log(wb.official_fx.where(wb.official_fx>0)).groupby(wb.iso3).diff().where(gap)
    d=cy.merge(wb,on=['iso3','year'],how='inner',validate='one_to_one');out=[]
    x=['z_'+c for c in ['inflation_outlook','hawkish_reaction','supply_narrative','fx_pressure','financial_stability']]+['inflation_lag1','gdp_growth_lag1']
    for outcome in ['inflation_lead1','fx_depreciation']:
        rr,_=fit(d,outcome,x,clusters=('country','year'));out += [dict(outcome=outcome,**r) for r in rr]
    save(pd.DataFrame(out),'macro_results');d.to_csv(ROOT/'derived/macro_panel.csv',index=False)
def benchmark_design(raw,lex):
    # Fresh estimation, no saved regression output enters the computation.
    d=add_method(raw,lex.set_index('doc_id').loc[raw.doc_id,CONCEPTS].reset_index(drop=True),'lex_')
    m=match_leaders(d,['lex_']);window=m[m.union45].copy();out=[]
    market=pd.read_csv(ROOT/'data/external/market_changes_cache.csv',parse_dates=['date','fed_date']);window=window.merge(market,on=['country','date','fed_date'],how='left',validate='one_to_one')
    us=pd.read_excel(ROOT/'data/external/USMPD.xlsx',sheet_name='Statements');us.columns=[str(c).strip() for c in us.columns];us['fed_date']=pd.to_datetime(us.Date).dt.normalize();us['abs_MP1']=pd.to_numeric(us.MP1,errors='coerce').abs()
    window=window.merge(us[['fed_date','abs_MP1']].drop_duplicates('fed_date'),on='fed_date',how='left',validate='many_to_one');window['z_abs_MP1']=z(window.abs_MP1)
    flags=set(pd.read_csv(ROOT/'audit/heading_date_candidates.csv').doc_id);window['event_weight']=1/window.groupby('fed_date').doc_id.transform('size')
    for c in CONCEPTS:
        y='z_lex_'+c;past='fed_zd_lex_'+c;ecb='ecb_zd_lex_'+c
        base=[past,ecb,'log_words'];controls=[x for x in ['d_vix','d_oil','d_dxy'] if x in window]
        window['language_x_mp1']=window[past]*window.z_abs_MP1
        for label,sample,x,weight in [('union_window',window,base,None),('both_leaders_45',window[window.days_since_fed.between(1,45)&window.days_since_ecb.between(1,45)],base,None),('market_controls',window,base+controls,None),('market_matched_no_controls',window.dropna(subset=controls),base,None),('absolute_policy_surprise',window,base+['z_abs_MP1'],None),('surprise_interaction',window,[past,'z_abs_MP1','language_x_mp1','log_words'],None),('drop_duplicate_foreign',window[~window.duplicate_text],base,None),('drop_heading_candidates',window[~window.doc_id.isin(flags)],base,None),('drop_2020_2022',window[~window.year.between(2020,2022)],base,None),('equal_events',window,base,'event_weight')]:
            rr,_=fit(sample,y,x,weight=weight);out += [dict(spec=label,concept=c,**r) for r in rr]
    save(pd.DataFrame(out),'union_window_results')
    # Country-group and cross-concept checks, recomputed from full lexical text.
    group=[];cross=[];contrasts=[]
    f=m[m.days_since_fed.between(1,45)].copy()
    for c in CONCEPTS:
        past='fed_zd_lex_'+c
        for name,mask in [('Advanced',f.advanced.eq(1)),('Emerging',f.advanced.eq(0)),('IT',f.inflation_targeter.eq(1)),('NonIT',f.inflation_targeter.eq(0)),('English',f.english_original.eq(1))]:
            rr,_=fit(f[mask],'z_lex_'+c,[past,'log_words']);group += [dict(group=name,concept=c,**r) for r in rr]
        for attr in ['advanced','inflation_targeter']:
            f['interaction']=f[past]*f[attr];rr,_=fit(f,'z_lex_'+c,[past,'interaction','log_words']);contrasts += [dict(group=attr,concept=c,**r) for r in rr if r['term']=='interaction']
        for dst in CONCEPTS:
            rr,_=fit(f,'z_lex_'+dst,[past,'log_words']);cross += [dict(source=c,destination=dst,**r) for r in rr if r['term']==past]
    save(pd.DataFrame(group),'follower_groups');save(bh(pd.DataFrame(contrasts),[],family_size=14),'follower_group_contrasts');save(bh(pd.DataFrame(cross),[],family_size=49),'cross_concept_results')
def run():
    raw=pd.read_json(ROOT/'derived/corpus.jsonl',lines=True);raw.date=pd.to_datetime(raw.date);lex=pd.read_csv(ROOT/'derived/lexical_scores.csv')
    cy=annual(raw,lex);lib=json.loads((ROOT/'data/exemplars.json').read_text());v,w=fit_lexical(lib);markets(v,w);macro(cy);benchmark_design(raw,lex)
    print('Supplementary annual, market, macro and union-window estimates complete',flush=True)
if __name__=='__main__':run()


### Granularity ties

In [ ]:
%%writefile code/granularity_ties.py
"""Exact empirical-grid allocation audit and lexical-tie sensitivity.

The baseline map breaks exact lexical ties by date then document id. Since exact
marginal matching can split such ties across grid values, five prespecified hash
orders audit that otherwise arbitrary part of the experiment. Lexical ranking
between unequal scores is always preserved. No seed is selected by its result.
"""
import pandas as pd,numpy as np
from data import ROOT,CONCEPTS,sha
from estimation import add_method,match_leaders,fit,bh

def run():
    d=pd.read_csv(ROOT/'derived/diagnostic_scores.csv',parse_dates=['date']).sort_values(['country','date']).reset_index(drop=True)
    mapping=[];summary=[];estimates=[]
    for c in CONCEPTS:
        for bank,g in d.groupby('country'):
            q=g.sort_values(['lex_'+c,'date','doc_id'],kind='stable').copy()
            q['mapped']=np.sort(q['old_'+c].to_numpy())
            blocks=q.groupby('lex_'+c).agg(n=('doc_id','size'),grid_values=('mapped','nunique'))
            split=blocks[(blocks.n>1)&(blocks.grid_values>1)]
            summary.append(dict(country=bank,concept=c,n_documents=len(g),n_lexical_values=g['lex_'+c].nunique(),n_lexical_tie_blocks=int((blocks.n>1).sum()),split_tie_blocks=len(split),documents_in_split_ties=int(split.n.sum())))
            mapping.extend(dict(doc_id=r.doc_id,country=bank,concept=c,lexical=getattr(r,'lex_'+c),mapped_score=r.mapped) for r in q.itertuples())
    pd.DataFrame(summary).to_csv(ROOT/'audit/granularity_tie_allocation.csv',index=False)
    pd.DataFrame(mapping).to_csv(ROOT/'audit/granularity_baseline_map.csv',index=False)
    for seed in [11,29,47,71,101]:
        mapped=pd.DataFrame(index=d.index,columns=CONCEPTS,dtype=float)
        for c in CONCEPTS:
            for bank,g in d.groupby('country'):
                q=g.assign(tie_hash=g.doc_id.map(lambda x:sha(f'grid-ties-{seed}|{x}')))
                order=q.sort_values(['lex_'+c,'tie_hash'],kind='stable').index
                mapped.loc[order,c]=np.sort(g['old_'+c].to_numpy())
        model=add_method(d,mapped,'grid_');mat=match_leaders(model,['lex_','old_','emb_','grid_']);mat=mat[mat.union45]
        for c in CONCEPTS:
            required=[side+p+c for side in ['z_','fed_zd_','ecb_zd_'] for p in ['lex_','old_','emb_','grid_']]
            sample=mat.dropna(subset=required)
            for label,y in [('grid_leader_only','z_lex_'),('grid_both','z_grid_')]:
                rr,_=fit(sample,y+c,['fed_zd_grid_'+c,'ecb_zd_grid_'+c,'log_words'])
                estimates.extend(dict(seed=seed,concept=c,method=label,**r) for r in rr if r['term'].startswith('fed_'))
    bh(pd.DataFrame(estimates),['seed','method'],family_size=7).to_csv(ROOT/'results/granularity_tie_sensitivity.csv',index=False)
    print('Five prespecified lexical-tie allocations complete; all unequal-score ranks preserved.')
if __name__=='__main__':run()


### Attention events

In [ ]:
%%writefile code/attention_events.py
"""Predefined descriptive attention windows; equal jurisdiction weights, no event-study causality."""
import pandas as pd,numpy as np
from data import ROOT,CONCEPTS,ADVANCED,read_jsonl
from analysis import save
WINDOWS=[('financial_stability','2008-09','2008-01-01','2009-12-31'),('financial_stability','March 2023','2023-03-01','2023-03-31'),('supply_narrative','2021-22','2021-01-01','2022-12-31')]
def run():
    s=pd.read_csv(ROOT/'derived/continuous_scores_long.csv',parse_dates=['date'])
    ids={r['doc_id'] for r in read_jsonl(ROOT/'derived/corpus.jsonl')}
    s=s[(s.kind=='level')&s.doc_id.isin(ids)]
    for draw in [1,2]:
        if set(s.loc[s.draw==draw,'doc_id'])!=ids:raise RuntimeError('Attention diagnostics require complete corpus draws')
    doc=s.groupby(['doc_id','country','date','concept']).agg(coverage=('coverage','mean'),available_draws=('draw','nunique')).reset_index()
    rows=[]
    for concept,label,start,end in WINDOWS:
        period=doc[(doc.concept==concept)&doc.date.between(start,end)]
        bybank=period.groupby('country').coverage.mean()
        rows.append(dict(concept=concept,period=label,start=start,end=end,n_documents=len(period),n_countries=len(bybank),equal_country_mean_coverage=bybank.mean(),median_country_coverage=bybank.median(),statement_weighted_mean_coverage=period.coverage.mean()))
    save(pd.DataFrame(rows),'attention_named_periods')
    doc['year']=doc.date.dt.year;doc['group']=np.where(doc.country.isin(ADVANCED),'Advanced','Other jurisdictions')
    cy=doc.groupby(['country','year','concept','group']).coverage.mean().reset_index()
    save(cy.groupby(['year','concept','group']).agg(mean_coverage=('coverage','mean'),countries=('country','nunique')).reset_index(),'attention_annual_by_group')
    print('Predefined attention windows and jurisdiction groups complete.')
if __name__=='__main__':run()


### Representation diagnostics

In [ ]:
%%writefile code/representation_diagnostics.py
"""Full-text method agreement and paired coefficient-difference inference."""
from data import *
from estimation import fit,bh
from analysis import save
from scipy import stats

def run():
    d=pd.read_csv(ROOT/'derived/analysis_statement_scores.csv',parse_dates=['date'])
    raw=read_jsonl(ROOT/'derived/corpus.jsonl');texts={r['doc_id']:r['text'] for r in raw}
    dic=[]
    for doc in d.doc_id:
        t=re.findall(r"[A-Za-z']+",texts[doc].lower());n=max(1,len(t))
        rate=lambda vocab:1000*sum(x in vocab for x in t)/n
        dic.append(dict(doc_id=doc,dict_hd=rate(HAWKISH_WORDS)-rate(DOVISH_WORDS),dict_tone=rate(LM_POS)-rate(LM_NEG)))
    dic=pd.DataFrame(dic);dic.to_csv(ROOT/'derived/dictionary_scores.csv',index=False);d=d.merge(dic,on='doc_id',validate='one_to_one')
    coverage=pd.read_csv(ROOT/'derived/continuous_scores_long.csv')
    coverage=coverage[(coverage.kind=='level')&coverage.doc_id.isin(d.doc_id)];repeats=[]
    for c,g in coverage.groupby('concept'):
        wide=g.pivot(index='doc_id',columns='draw',values='coverage')
        repeats.append(dict(concept=c,n=len(wide),correlation=wide[1].corr(wide[2]),mae=(wide[1]-wide[2]).abs().mean(),absence_agreement=wide[1].eq(0).eq(wide[2].eq(0)).mean()))
    save(pd.DataFrame(repeats),'coverage_repeatability')
    out=[]
    for c in CONCEPTS:
        for method,col in [('Continuous LLM','llm_'+c),('Full embeddings','emb_'+c),('Hawk-dove word proxy','dict_hd'),('Tone word proxy','dict_tone')]:
            for sample,g in [('Pooled',d),*list(d.groupby('country'))]:
                a=g[['lex_'+c,col]].dropna()
                out.append(dict(concept=c,comparison=method,sample=sample,n=len(a),correlation=a.iloc[:,0].corr(a.iloc[:,1]) if len(a)>2 else np.nan))
    save(pd.DataFrame(out),'measurement_correlations')
    m=pd.read_csv(ROOT/'derived/analysis_matched_statements.csv',parse_dates=['date','fed_date','future_date']);m=m[m.days_since_fed.between(1,45)&m.days_until_fed.between(1,45)]
    differences=[]
    for c in CONCEPTS:
        names=['lex_','llm_','emb_','pair_'];y='z_lex_'+c
        sample=m.dropna(subset=[y,'log_words','country','year','fed_date',*['fed_zd_'+p+c for p in names]]).copy()
        fitted={}
        for pre in names:
            term='fed_zd_'+pre+c;rr,obj=fit(sample,y,[term,'log_words'])
            row=next(r for r in rr if r['term']==term)
            if obj is not None and row['status']=='identified':fitted[pre]=(row,obj,term)
        for pre in names[1:]:
            base=dict(concept=c,comparison='lex_ minus '+pre,n=len(sample))
            if 'lex_' not in fitted or pre not in fitted:
                differences.append(dict(**base,status='unidentified',estimate=np.nan,std_error=np.nan,p_value=np.nan));continue
            r0,f0,t0=fitted['lex_'];r1,f1,t1=fitted[pre]
            assert f0['data'].index.equals(f1['data'].index)
            j0=f0['X'].columns.get_loc(t0);j1=f1['X'].columns.get_loc(t1)
            # Observation influence functions include each model's own design/bread.
            i0=(f0['X'].to_numpy()@f0['bread'][:,j0])*f0['resid']
            i1=(f1['X'].to_numpy()@f1['bread'][:,j1])*f1['resid'];infl=i0-i1
            variance=0.;counts=[];n=len(sample);p=f0['X'].shape[1]
            assert p==f1['X'].shape[1]
            for cols,sgn in [(['country'],1),(['fed_date'],1),(['country','fed_date'],-1)]:
                groups=pd.MultiIndex.from_frame(sample[cols]);codes,labels=pd.factorize(groups);g=len(labels);sums=np.bincount(codes,weights=infl)
                variance+=sgn*g/max(1,g-1)*(n-1)/max(1,n-p)*np.sum(sums**2);counts.append(g)
            delta=r0['estimate']-r1['estimate'];se=np.sqrt(variance) if variance>0 else np.nan;df=min(counts[:2])-1;critical=stats.t.ppf(.975,df)
            differences.append(dict(**base,status='estimated',estimate=delta,std_error=se,p_value=2*stats.t.sf(abs(delta/se),df) if pd.notna(se) else np.nan,ci_low=delta-critical*se,ci_high=delta+critical*se,lexical_estimate=r0['estimate'],alternative_estimate=r1['estimate'],opposite_point_signs=r0['estimate']*r1['estimate']<0,df_t=df))
    save(bh(pd.DataFrame(differences),[],family_size=21),'leader_coefficient_differences')
    from sklearn.metrics import cohen_kappa_score
    pairs=read_jsonl(ROOT/'derived/pair_annotations.jsonl');frame=pd.DataFrame([dict(pair_id=r['pair_id'],country=r['country'],source=r['source'],draw=r['draw'],edit_type=r['annotation']['edit_type']) for r in pairs]);rows=[]
    for (bank,source),g in frame.groupby(['country','source']):
        wide=g.pivot(index='pair_id',columns='draw',values='edit_type')
        rows.append(dict(country=bank,source=source,n=len(wide),exact_agreement=wide[1].eq(wide[2]).mean(),kappa=cohen_kappa_score(wide[1],wide[2])))
    save(pd.DataFrame(rows),'pair_edit_repeatability')
    benchmark=pd.read_csv(ROOT/'derived/diagnostic_scores.csv').set_index('doc_id');zeros=[]
    for bank,g in d.groupby('country'):
        if bank not in ['US','ECB']:continue
        g=g.sort_values('date')
        for c in CONCEPTS:
            coarse=benchmark.reindex(g.doc_id)['old_'+c].diff();coarse.index=g.index
            series={'Benchmark coarse':coarse,'TF-IDF':g['lex_'+c].diff(),'Full embeddings':g['emb_'+c].diff(),'Continuous LLM':g['llm_'+c].diff(),'Pair draw 1':g['pair1_'+c],'Pair draw 2':g['pair2_'+c]}
            common=pd.concat(series,axis=1).dropna()
            for method in series:zeros.append(dict(country=bank,concept=c,method=method,n_common_pairs=len(common),zero_changes=int(common[method].eq(0).sum()),zero_rate=common[method].eq(0).mean()))
    save(pd.DataFrame(zeros),'strict_common_leader_zero_rates')
    print('Cross-method correlations and paired leader-coefficient contrasts complete.')
if __name__=='__main__':run()


### Inference sensitivity

In [ ]:
%%writefile code/inference_sensitivity.py
"""Report alternative clustering after a non-positive two-way variance was detected.

This is a declared numerical/dependence sensitivity applied to every concept and
method, not a replacement for the raw two-way covariance or a significance filter.
No covariance eigenvalues or negative diagonals are silently repaired.
"""
import pandas as pd
from data import ROOT,CONCEPTS
from estimation import fit,bh
from analysis import save

def run():
    m=pd.read_csv(ROOT/'derived/analysis_matched_statements.csv',parse_dates=['date','fed_date','future_date'])
    m=m[m.days_since_fed.between(1,45)&m.days_until_fed.between(1,45)].copy();m['month']=m.date.dt.to_period('M').astype(str)
    rows=[]
    for method,outcome,leader in [('TF-IDF','lex_','lex_'),('Continuous LLM','llm_','llm_'),('Full embeddings','emb_','emb_'),('Pairwise leader / lexical outcome','lex_','pair_')]:
        for c in CONCEPTS:
            y='z_'+outcome+c;p='fed_zd_'+leader+c;f='future_zd_'+leader+c;sample=m.dropna(subset=[y,p,f,'log_words'])
            for label,clusters in [('country_only',('country',)),('Fed_release_only',('fed_date',)),('country_and_month',('country','month'))]:
                for spec,x,contrast in [('preceding_only',[p,'log_words'],None),('joint',[p,f,'log_words'],{'past_minus_future':{p:1,f:-1}})]:
                    rr,_=fit(sample,y,x,clusters=clusters,contrasts=contrast)
                    for r in rr:
                        if r['term']==('past_minus_future' if spec=='joint' else p):rows.append(dict(method=method,concept=c,clustering=label,spec=spec,**r))
    save(bh(pd.DataFrame(rows),['method','clustering','spec'],family_size=7),'inference_clustering_sensitivity')
    print('All-method clustering sensitivity complete; raw two-way estimates retained.')
if __name__=='__main__':run()


### Identification

In [ ]:
%%writefile code/identification.py
"""Counterexample: past-minus-future differences do not generally cancel common news.
g_t = rho*g_(t-1)+epsilon_t; foreign z_t=g_t with ZERO communication transmission.
Raw slopes on (g_t-g_(t-1),g_(t+1)-g_t) are +/-1/(3-rho).
For unit-SD outcome and unit-SD regressors, multiply by sqrt(2*(1-rho)).
"""
import numpy as np,pandas as pd
from data import ROOT
def run(seed=23019,n=200000):
    rng=np.random.default_rng(seed);rows=[]
    for rho in [0,.25,.5,.75,.9,.95]:
        g=np.zeros(n+2001);noise=rng.normal(size=len(g))*np.sqrt(1-rho*rho)
        for i in range(1,len(g)):g[i]=rho*g[i-1]+noise[i]
        g=g[2000:];y=g[1:-1];past=g[1:-1]-g[:-2];future=g[2:]-g[1:-1]
        X=np.column_stack([np.ones(len(y)),past/past.std(),future/future.std()]);b=np.linalg.lstsq(X,y/y.std(),rcond=None)[0]
        expected=np.sqrt(2*(1-rho))/(3-rho)
        rows.append(dict(rho=rho,true_transmission=0,expected_past=expected,expected_future=-expected,expected_difference=2*expected,
            simulated_past=b[1],simulated_future=b[2],simulated_difference=b[1]-b[2],n=len(y),seed=seed))
    df=pd.DataFrame(rows);df.to_csv(ROOT/'results/common_news_counterexample.csv',index=False)
    assert np.max(np.abs(df.simulated_difference-df.expected_difference))<.015
    print(df.to_string(index=False))
if __name__=='__main__':run()


### Saved measurements

In [ ]:
%%writefile code/saved_measurements.py
"""Checks and reconstruction summaries from exported measurement records, offline."""
import json,re
import pandas as pd
from data import ROOT,read_jsonl
def leader_audit():
    scores={}
    for r in read_jsonl(ROOT/'derived/pair_annotations.jsonl'):scores.setdefault(r['pair_id'],[]).append(r)
    flags=pd.read_csv(ROOT/'audit/leader_pair_flags.csv').set_index('pair_id').boilerplate_only_rule.to_dict()
    rows=[];checks=[]
    for pair in read_jsonl(ROOT/'audit/leader_redlines.jsonl'):
        draws=sorted(scores.get(pair['pair_id'],[]),key=lambda r:r['draw'])
        assert [r['draw'] for r in draws]==[1,2],pair['pair_id']
        for r in draws:
            assert r['text_hash']==pair['text_hash'] and r['old_text_hash']==pair['old_text_hash'] and r['redline_hash']==pair['redline_hash']
            numeric=[a['change'] for a in r['annotation']['concepts'].values() if a['change'] is not None]
            checks.append(dict(pair_id=pair['pair_id'],country=pair['country'],source=pair['source'],draw=r['draw'],identical_source=pair['identical'],boilerplate_rule=bool(flags[pair['pair_id']]),edit_type=r['annotation']['edit_type'],nonzero_changes=sum(x!=0 for x in numeric),numeric_changes=len(numeric)))
        rows.append(dict(**pair,boilerplate_only_rule=bool(flags[pair['pair_id']]),draws=draws))
    with (ROOT/'audit/leader_change_audit.jsonl').open('w') as f:
        for r in rows:f.write(json.dumps(r,ensure_ascii=False)+'\n')
    pd.DataFrame(checks).to_csv(ROOT/'audit/pair_classification_consistency.csv',index=False)
    print('Joined full redlines and two draws for',len(rows),'leader pairs.')

def reconstruction_summary():
    rr=pd.read_csv(ROOT/'audit/reconstruction_backtranslation.csv')
    rr['absolute_error']=(rr.original_score-rr.reconstructed_score).abs()
    rr['absence_preserved']=(rr.original_coverage.eq(0)==rr.reconstructed_coverage.eq(0))
    rr.groupby('concept').agg(n=('doc_id','size'),score_pairs=('absolute_error','count'),mae=('absolute_error','mean'),absence_agreement=('absence_preserved','mean')).reset_index().to_csv(ROOT/'results/reconstruction_summary.csv',index=False)
    anchors=read_jsonl(ROOT/'audit/reconstruction_anchor_audit.jsonl')
    columns=['concept','target_score','draw','recovered_score','coverage','reconstruction_key','backscore_key']
    pd.DataFrame([{key:row[key] for key in columns} for row in anchors],columns=columns).to_csv(ROOT/'results/reconstruction_separation.csv',index=False)


### Verification

In [ ]:
%%writefile code/verification.py
"""Independent numerical and scientific invariants, run entirely offline."""
import json
import numpy as np,pandas as pd
from data import ROOT,CONCEPTS,read_jsonl
from estimation import fit,bh

def run():
    from statsmodels.regression.linear_model import OLS,WLS
    from statsmodels.stats.sandwich_covariance import cov_cluster_2groups
    rng=np.random.default_rng(627144);n=1200
    d=pd.DataFrame({'country':np.repeat(np.arange(30),40),'year':np.tile(np.repeat(np.arange(4),10),30),'fed_date':np.tile(np.arange(40),30),'x':rng.normal(size=n),'future':rng.normal(size=n),'weight':rng.uniform(.3,2,n)})
    d['y']=.3*d.x-.2*d.future+rng.normal(size=n)+rng.normal(size=30)[d.country]
    for weight in [None,'weight']:
        rows,f=fit(d,'y',['x','future'],weight=weight,contrasts={'difference':{'x':1,'future':-1}})
        reference=(OLS(d.y,f['X']) if weight is None else WLS(d.y,f['X'],weights=d.weight)).fit()
        covariance=cov_cluster_2groups(reference,d.country,d.fed_date)[0]
        np.testing.assert_allclose(f['beta'],reference.params,rtol=1e-8,atol=1e-10)
        np.testing.assert_allclose(f['cov'],covariance,rtol=1e-8,atol=1e-10)
        np.testing.assert_allclose(rows[0]['r_squared'],reference.rsquared,atol=1e-10)
        v=np.zeros(len(f['beta']));v[f['X'].columns.get_loc('x')]=1;v[f['X'].columns.get_loc('future')]=-1
        np.testing.assert_allclose(rows[-1]['std_error'],np.sqrt(v@covariance@v),atol=1e-10)
    main=pd.read_csv(ROOT/'results/main_results.csv')
    for (_,c),g in main.groupby(['method','concept']):
        assert g[g.leg=='past'].n.nunique()==1,(c,'same-sample preceding design')
    phrases=pd.read_csv(ROOT/'results/phrase_adoption.csv')
    good=phrases.estimate.notna()
    np.testing.assert_allclose(phrases.loc[good,'past_mean_pp']-phrases.loc[good,'future_mean_pp'],phrases.loc[good,'estimate'],atol=1e-9)
    corpus=read_jsonl(ROOT/'derived/corpus.jsonl');assert len(corpus)==6692
    status=json.loads((ROOT/'audit/scoring_completion.json').read_text());assert status['missing_physical_jobs']==0
    audit=read_jsonl(ROOT/'audit/leader_change_audit.jsonl');assert len(audit)==755 and all(len(r['draws'])==2 for r in audit)
    assert all(r['draws'][0]['response_id']!=r['draws'][1]['response_id'] for r in audit)
    # Unidentified tests stay in the planned family as non-rejections.
    small=bh(pd.DataFrame({'p_value':[.01,.03,np.nan]}),[],family_size=3)
    np.testing.assert_allclose(small.q_value.iloc[:2],[.03,.045])
    result={'status':'passed','checks':['independent OLS/WLS coefficients and two-way covariance','weighted R-squared','joint contrast covariance','same-sample preceding design','paired phrase means','full corpus and scoring coverage','755 complete leader audit records and distinct repeated response ids','BH family includes unidentified tests as non-rejections']}
    (ROOT/'audit/verification.json').write_text(json.dumps(result,indent=2));print(json.dumps(result,indent=2))
if __name__=='__main__':run()


### Exhibits

In [ ]:
%%writefile code/exhibits.py
"""Numerical builders for the tables retained in the September 10 manuscript."""
import json, re
from pathlib import Path
import numpy as np, pandas as pd
from data import ROOT, CONCEPTS, LABELS, COUNTRY_NAME, read_jsonl
SHORT = {'activity_outlook': 'Activity', 'inflation_outlook': 'Inflation', 'hawkish_reaction': 'Hawkish', 'supply_narrative': 'Supply', 'uncertainty': 'Uncertainty', 'fx_pressure': 'Exchange rate', 'financial_stability': 'Financial stability', 'disagreement': 'Disagreement'}
METHODS = ['TF-IDF', 'Continuous LLM', 'Full embeddings', 'Pairwise leader / lexical outcome']
METHOD_NAMES = ['TF--IDF', 'Direct LLM', 'Embeddings', 'Pairwise leader']
INFER = JOINT = ''
CATALOG = {}

def read(name):
    return pd.read_csv(ROOT / 'results' / f'{name}.csv')

def esc(s):
    return str(s).replace('&', '\\&').replace('%', '\\%').replace('_', '\\_').replace('#', '\\#')

def num(x, d=3):
    if pd.isna(x) or not np.isfinite(float(x)):
        return 'n.a.'
    if abs(float(x)) < 0.5 * 10 ** (-d):
        x = 0
    return f'{x:.{d}f}'

def prob(x):
    return '$<.001$' if pd.notna(x) and x < 0.001 else num(x)

def integer(x):
    return 'n.a.' if pd.isna(x) else f'{int(x):,}'

def coef(r, showq=False, shown=False, digits=3):
    if r is None or pd.isna(r.get('estimate')):
        return '\\shortstack[r]{n.a.\\\\{\\scriptsize unavailable}}'
    lines = [num(r['estimate'], digits), f"({num(r.get('std_error'), digits)})"]
    if showq or shown:
        terms = []
        if showq:
            terms.append('$q=' + num(r.get('q_value'), digits) + '$')
        if shown:
            terms.append('$n=' + str(int(r['n'])) + '$')
        lines.append('{\\scriptsize ' + ', '.join(terms) + '}')
    return '\\shortstack[r]{' + '\\\\'.join(lines) + '}'

def pick(d, **kwargs):
    s = d
    for k, v in kwargs.items():
        s = s[s[k] == v]
    return s.iloc[0].to_dict() if len(s) else None

def matrix_table(key, title, label, d, column, values, headers, notes, filterterm=None, showq=False, shown=False):
    if filterterm is not None:
        d = d[d.term.map(filterterm)]
    rows = []
    for c in CONCEPTS:
        rows.append([SHORT[c], *[coef(pick(d, concept=c, **{column: v}), showq, shown) for v in values]])
    return write_table(key, title, label, ['Concept', *headers], rows, d, notes)

def _numeric_table(key):
    if key in ['tab:methods', 'tab:preceding']:
        d = read('main_results')
        rows = []
        for title, spec, leg in [('A. Preceding Fed change, entered alone', 'preceding_only', 'past'), ('B. Preceding minus following Fed change, entered jointly', 'joint', 'difference')]:
            rows.append('\\multicolumn{5}{l}{\\textit{' + title + '}} \\\\ \\addlinespace[3pt]')
            for c in CONCEPTS:
                rows.append([SHORT[c], *[coef(pick(d, method=m, concept=c, spec=spec, leg=leg), True, True) for m in METHODS]])
            rows.append('\\addlinespace[5pt]')
        return write_table(key, '', '', ['Concept', *METHOD_NAMES], rows, d, '', size='footnotesize')
    if key in ['tab:phrases', 'symmetric_phrase_support']:
        design = 'past_window' if key == 'tab:phrases' else 'symmetric_window'
        d = read('phrase_adoption')
        d = d[d.design == design]
        rows = []
        splits = ['all', 'names_Fed', 'does_not_name_Fed', 'Official_English', 'Database_translated']
        names = ['All statements', 'Names the Fed', 'Does not name Fed', 'Official English', 'Database translated']
        for variant, title in [('all_novel', 'All observed novel phrases'), ('meaningful_novel', 'LLM-screened meaningful phrases')]:
            rows.append('\\multicolumn{6}{l}{\\textit{' + title + '}} \\\\ \\addlinespace')
            for split, label in zip(splits, names):
                rows.append([label, *[coef(pick(d, variant=variant, split=split, horizon=h)) for h in [7, 14, 21, 30, 45]]])
        return write_table(key, '', '', ['Sample', '7 days', '14 days', '21 days', '30 days', '45 days'], rows, d, '', size='footnotesize')
    if key == 'tab:measure':
        d = read('measurement_correlations')
        d = d[d['sample'] == 'Pooled']
        rows = []
        for c in CONCEPTS:
            rows.append([SHORT[c], *[num(pick(d, concept=c, comparison=m)['correlation']) for m in ['Continuous LLM', 'Full embeddings', 'Hawk-dove word proxy', 'Tone word proxy']], integer(pick(d, concept=c, comparison='Continuous LLM')['n'])])
        return write_table(key, '', '', ['Concept', 'Direct LLM', 'Embeddings', 'Hawk--dove', 'Tone', 'LLM $n$'], rows, d, '')
    if key == 'tab:sync':
        d = read('annual_comovement')
        vals = ['All', 'Excl. US/ECB', 'Advanced', 'Other', 'English']
        return write_table(key, '', '', ['Concept', 'All', 'No US/ECB', 'Advanced', 'Other', 'Six-bank core'], [[SHORT[c], *[num(pick(d, concept=c, sample=v)['projection_share'], 1) for v in vals]] for c in CONCEPTS], d, '')
    if key == 'tab:balanced':
        d = read('annual_balanced')
        return write_table(key, '', '', ['Concept', 'All: projection', 'All: joint FE', 'Balanced: projection', 'Balanced: joint FE'], [[SHORT[c], *[num(pick(d, concept=c, sample=s)[col], 1) for s, col in [('All', 'projection_share'), ('All', 'joint_fe_partial_r2'), ('Balanced', 'projection_share'), ('Balanced', 'joint_fe_partial_r2')]]] for c in CONCEPTS], d, '')
    if key == 'tab:placebo':
        d = read('main_results')
        d = d[(d.method == 'TF-IDF') & (d.spec == 'joint')]
        return matrix_table(key, '', '', d, 'leg', ['past', 'future', 'difference'], ['Preceding', 'Following', 'Difference'], '', showq=False)
    if key == 'tab:contrasts':
        d = read('follower_group_contrasts')
        return matrix_table(key, '', '', d, 'group', ['advanced', 'inflation_targeter'], ['Advanced minus other', 'Targeter minus other'], '', showq=False)
    if key in ['tab:minutes', 'tab:ecbmarket']:
        source = 'minutes' if key == 'tab:minutes' else 'ecb'
        d = read('market_results')
        d = d[d.source == source]
        outcomes = ['UST2Y', 'UST10Y', 'SP500', 'TIPS5Y'] if source == 'minutes' else ['OIS_2Y', 'DE10Y', 'STOXX50', 'EURUSD']
        rows = []
        for c in CONCEPTS[:5] + ['disagreement']:
            rows.append([SHORT[c], *[coef(pick(d, term='zd_' + c, outcome=o), showq=c == 'hawkish_reaction') for o in outcomes]])
        rows.append(['Observations', *[integer(d[d.outcome == o].n.iloc[0]) for o in outcomes]])
        return write_table(key, '', '', ['Concept', *outcomes], rows, d, '')
    if key == 'tab:annual-leaders':
        d = read('annual_leader_follower')
        rows = [[SHORT[c], *[coef(pick(d, concept=c, term=p + c)) for p in ['lag_', 'fed_lag_', 'ecb_lag_']]] for c in CONCEPTS]
        return write_table(key, '', '', ['Concept', 'Own lag', 'Fed lag', 'ECB lag'], rows, d, '')
    if key == 'tab:macro':
        d = read('macro_results')
        terms = ['z_' + c for c in ['inflation_outlook', 'hawkish_reaction', 'supply_narrative', 'fx_pressure', 'financial_stability']] + ['inflation_lag1', 'gdp_growth_lag1']
        rows = []
        for t in terms:
            rows.append([SHORT.get(t.replace('z_', ''), {'inflation_lag1': 'Lagged inflation', 'gdp_growth_lag1': 'Lagged GDP growth'}.get(t, t)), *[coef(pick(d, term=t, outcome=o)) for o in ['inflation_lead1', 'fx_depreciation']]])
        return write_table(key, '', '', ['Variable', 'Next-year inflation', 'Same-year depreciation'], rows, d, '')
    if key == 'tab:reliability':
        d = read('test_retest_reliability')
        rows = []
        for c in CONCEPTS:
            a = pick(d, kind='level', country='ALL', concept=c)
            b = pick(d, kind='pair', country='US', concept=c)
            rows.append([SHORT[c], integer(a['n_paired']), num(a['correlation']), num(a['mae'], 1), integer(b['n_paired']), num(b['correlation']), num(b['mae'], 1)])
        return write_table(key, '', '', ['Concept', 'Level $n$', 'Level $r$', 'Level MAE', 'Fed pair $n$', 'Pair $r$', 'Pair MAE'], rows, d, '')
    if key == 'tab:ecbrepair':
        d = read('ecb_source_repair_same_dates')
        d = d[d.leader == 'ECB']
        return matrix_table(key, '', '', d, 'spec', ['raw_same_dates', 'full_intro_same_dates', 'raw_excluding_boilerplate', 'full_intro_excluding_boilerplate', 'raw_same_predecessor', 'full_intro_same_predecessor'], ['Raw', 'Full intro.', 'Raw: no boiler.', 'Full: no boiler.', 'Raw: same prior', 'Full: same prior'], '')
    if key == 'tab:coefficientdifferences':
        d = read('leader_coefficient_differences')
        return matrix_table(key, '', '', d, 'comparison', ['lex_ minus llm_', 'lex_ minus emb_', 'lex_ minus pair_'], ['Lexical minus LLM', 'Lexical minus embeddings', 'Lexical minus pair'], '', showq=False)
    if key == 'tab:reconstruction':
        d = read('reconstruction_summary')
        s = read('reconstruction_separation').groupby(['concept', 'target_score']).recovered_score.mean().unstack()
        raw = d.merge(s, on='concept')
        rows = [[SHORT[r.concept], integer(r.score_pairs), num(r.mae, 1), num(100 * r.absence_agreement, 1), *[num(s.loc[r.concept, k], 1) for k in [10, 50, 90]]] for r in d.itertuples()]
        return write_table(key, '', '', ['Concept', 'Score pairs', 'MAE', 'Absence (\\%)', 'Target 10', 'Target 50', 'Target 90'], rows, raw, '', size='footnotesize')
    if key == 'tab:claims':
        d = read('claim_alignment_horizons')
        d = d[d.evidence == 'verified_quotes']
        rows = []
        for draw in [1, 2]:
            rows.append('\\multicolumn{6}{l}{\\textit{Draw ' + str(draw) + '}} \\\\ \\addlinespace')
            for scope in ['domestic', 'US_reporting']:
                for c in ['activity', 'inflation']:
                    rows.append([esc(scope.replace('_', ' ') + ' / ' + c), *[coef(pick(d, draw=draw, scope=scope, concept=c, horizon=h), digits=2) for h in [7, 14, 21, 30, 45]]])
        return write_table(key, '', '', ['Scope / claim', '7 days', '14 days', '21 days', '30 days', '45 days'], rows, d, '', size='footnotesize')
    if key == 'tab:clustering':
        d = read('inference_clustering_sensitivity')
        d = d[(d.concept == 'hawkish_reaction') & (d.spec == 'joint')]
        base = read('main_results')
        base = base[(base.concept == 'hawkish_reaction') & (base.spec == 'joint') & (base.leg == 'difference')].copy()
        base['clustering'] = 'country_and_Fed'
        d = pd.concat([base, d], ignore_index=True)
        rows = [[label, *[coef(pick(d, method=m, clustering=c)) for c in ['country_and_Fed', 'country_only', 'Fed_release_only', 'country_and_month']]] for m, label in zip(METHODS, METHOD_NAMES)]
        return write_table(key, '', '', ['Method', 'Country + Fed', 'Country only', 'Fed only', 'Country + month'], rows, d, '', size='footnotesize')
    if key == 'tab:counterexample':
        d = read('common_news_counterexample')
        rows = [[num(r.rho, 2), num(r.expected_past), num(r.expected_future), num(r.expected_difference), num(r.simulated_difference)] for r in d.itertuples()]
        return write_table(key, '', '', ['$\\rho$', 'Preceding', 'Following', 'Difference', 'Simulated difference'], rows, d, '')
    if key == 'tab:precision':
        d = read('precision_and_equivalence')
        d = d[(d.spec == 'joint') & (d.leg == 'difference')]
        rows = []
        for m, label in zip(METHODS, METHOD_NAMES):
            rows.append('\\multicolumn{5}{l}{\\textit{' + label + '}} \\\\')
            for c in CONCEPTS:
                r = pick(d, method=m, concept=c)
                rows.append([SHORT[c], num(r['ci90_low']) + ' to ' + num(r['ci90_high']), 'Yes' if r['equivalent_within_005'] else 'No', 'Yes' if r['equivalent_within_010'] else 'No', num(r['approx_80pct_mde'])])
        return write_table(key, '', '', ['Concept', '90\\% interval', 'Within $\\pm .05$', 'Within $\\pm .10$', 'Approx. MDE'], rows, d, '', size='footnotesize')
    raise KeyError(key)

def _three_line_cell(r):
    """Main-text Table 1 cell as three aligned lines: estimate, (standard error), q and n."""
    if r is None or pd.isna(r.get('estimate')):
        return 'n.a.', '', '{\\scriptsize unavailable}'
    est = num(r['estimate'])
    est = '$-$' + est[1:] if est.startswith('-') else est
    se = '(' + num(r.get('std_error')) + ')'
    n = f"{int(r['n']):,}".replace(',', '{,}')
    return est, se, '{\\scriptsize $q=' + num(r.get('q_value')) + '$, $n=' + n + '$}'

def methods_three_line_rows(raw):
    """Rows of Table 1, one block of three lines per concept, separated by a small space."""
    labels = {'activity_outlook':'Activity outlook', 'inflation_outlook':'Inflation outlook',
              'hawkish_reaction':'Hawkish language', 'supply_narrative':'Supply attribution',
              'uncertainty':'Uncertainty', 'fx_pressure':'Exchange rate pressure',
              'financial_stability':'Financial stability'}
    lines = []
    for k, c in enumerate(CONCEPTS):
        cells = [_three_line_cell(pick(raw, method=m, concept=c)) for m in METHODS]
        lines.append(' & '.join([labels[c]] + [x[0] for x in cells]) + ' \\\\')
        lines.append(' & '.join([''] + [x[1] for x in cells]) + ' \\\\')
        lines.append(' & '.join([''] + [x[2] for x in cells]) + ' \\\\' + (' \\addlinespace[3pt]' if k < len(CONCEPTS) - 1 else ''))
    return lines

def polish_table(block, label):
    """Improve spacing only; retain every cell value, order, caption, and note."""
    stretch='1.16' if label=='tab:coverage-2' else '1.22'
    block=re.sub(r'(\\renewcommand\{\\arraystretch\}\{)[^}]+\}',lambda m:m[1]+stretch+'}',block)
    block=re.sub(r'(\\begin\{minipage\}\{[^\n]+?\}\\footnotesize)(?!\\raggedright)',r'\1\\raggedright',block)
    if label=='tab:methods':
        return block.replace(r'\addlinespace[3pt]',r'\addlinespace[4pt]')
    def space_rows(match):
        lines=match[2].splitlines()
        indices=[i for i,line in enumerate(lines) if line.rstrip().endswith(r'\\') and r'\multicolumn' not in line]
        for i in indices[:-1]:
            lines[i]=lines[i].rstrip()+r' \addlinespace[2pt]'
        return match[1]+'\n'.join(lines)+match[3]
    return re.sub(r'(\\midrule\n)(.*?)(\n\\bottomrule)',space_rows,block,count=1,flags=re.S)

def write_table(key, title, label, headers, rows, raw, notes, align=None, size='small'):
    publication_key = next((k for k, v in COMPUTATION_KEYS.items() if v == key))
    meta = TABLES[publication_key]
    output_key = PUBLISH_BY_LABEL[meta['label']]
    if key in ['tab:methods', 'tab:preceding']:
        rows = rows[10:17] if key == 'tab:methods' else rows[1:8]
        raw = raw[(raw.spec == ('joint' if key == 'tab:methods' else 'preceding_only')) & (raw.leg == ('difference' if key == 'tab:methods' else 'past'))]
    if key == 'tab:methods':
        rows = methods_three_line_rows(raw)
    if key == 'tab:preceding':
        rows = [[r[0], *[coef(pick(raw, concept=c, method=m)) if pick(raw, concept=c, method=m) and pd.notna(pick(raw, concept=c, method=m).get('estimate')) else 'n.a.' for m in METHODS]] for r, c in zip(rows, CONCEPTS)]
    if key == 'tab:precision':
        rows = [r if isinstance(r, str) or 'n.a.' not in r[1] else [r[0], 'n.a.', 'n.a.', 'n.a.', 'n.a.'] for r in rows]
    if key != 'tab:methods':
        rows = [r if isinstance(r, str) else [v.replace('\\shortstack[r]{n.a.\\\\{\\scriptsize unavailable}}', 'n.a.') for v in r] for r in rows]
    # The latest manuscript omits the unassessable FX display row; keep its full test family in the CSV.
    if key == 'tab:coefficientdifferences':
        rows = [r for r in rows if isinstance(r,str) or r[0] != 'Exchange rate']
    body = '\n'.join((r if isinstance(r, str) else ' & '.join(map(str, r)) + ' \\\\' for r in rows))
    block = re.sub('(\\\\midrule\\n).*?(\\\\bottomrule)', lambda m: m[1] + body + '\n' + m[2], meta['template'], count=1, flags=re.S)
    block=polish_table(block,meta['label'])
    TABLE_TEX[meta['label']]=block
    raw.to_csv(ROOT / 'output' / f'{output_key}.csv', index=False)
    CATALOG[output_key] = {k: v for k, v in meta.items() if k != 'template'}
    return raw

def make_table(publication_key):
    key = COMPUTATION_KEYS[publication_key]
    if key == 'tab:sensitivity':
        d = read('joint_sensitivity')
        specs = ['drop_duplicate_foreign', 'drop_heading_candidates', 'drop_2020_2022', 'own_previous_statement', 'equal_events']
        d = d[d.spec.isin(specs)]
        return matrix_table(key, '', '', d, 'spec', specs, [], '')
    if key == 'tab:coverage-2':
        raw = pd.DataFrame(read_jsonl(ROOT / 'derived/corpus.jsonl'))
        raw.date = pd.to_datetime(raw.date)
        d = raw.groupby('country').agg(statements=('doc_id', 'size'), first=('date', 'min'), last=('date', 'max'), median_words=('word_count', 'median')).reset_index()
        d['jurisdiction'] = d.country.map(COUNTRY_NAME)
        d = d.sort_values('jurisdiction').reset_index(drop=True)
        formatted = [[esc(r.jurisdiction), integer(r.statements), r.first.strftime('%Y-%m-%d'), r.last.strftime('%Y-%m-%d'), integer(r.median_words)] for r in d.itertuples()]
        rows = [formatted[i] + (formatted[i + 26] if i + 26 < len(formatted) else [''] * 5) for i in range(26)]
        return write_table(key, '', '', [], rows, d, '')
    if key == 'tab:spillover':
        d = read('union_window_results')
        d = d[d.spec.isin(['union_window', 'market_controls'])]
        rows = [[SHORT[c], *[coef(pick(d, concept=c, spec=spec, term=leader + '_zd_lex_' + c)) for spec in ['union_window', 'market_controls'] for leader in ['fed', 'ecb']]] for c in CONCEPTS]
        return write_table(key, '', '', [], rows, d, '')
    return _numeric_table(key)
TABLES = {'Table_1': {'title': 'Fed language associations across measurements.', 'label': 'tab:methods', 'csv': 'Table_1.csv'}, 'Table_2': {'title': 'Sensitivity of the lexical timing contrast.', 'label': 'tab:sensitivity', 'csv': 'Table_2.csv'}, 'Table_3': {'title': 'Literal phrase adoption across publication windows.', 'label': 'tab:phrases', 'csv': 'Table_3.csv'}, 'Table_A1': {'title': 'Jurisdiction coverage.', 'label': 'tab:coverage-2', 'csv': 'Table_A1.csv'}, 'Table_A2': {'title': 'Full text measurement agreement.', 'label': 'tab:measure', 'csv': 'Table_A2.csv'}, 'Table_A3': {'title': 'Annual lexical projection shares by country group.', 'label': 'tab:sync', 'csv': 'Table_A3.csv'}, 'Table_A4': {'title': 'Annual synchronization under two calculations and two samples.', 'label': 'tab:balanced', 'csv': 'Table_A4.csv'}, 'Table_A5': {'title': 'Fed and ECB associations in the union window, with and without financial controls.', 'label': 'tab:spillover', 'csv': 'Table_A5.csv'}, 'Table_A6': {'title': 'Preceding and following Fed changes, estimated jointly.', 'label': 'tab:placebo', 'csv': 'Table_A6.csv'}, 'Table_A7': {'title': 'Tests of follower-group differences.', 'label': 'tab:contrasts', 'csv': 'Table_A7.csv'}, 'Table_A8': {'title': 'Full text FOMC minutes and market changes.', 'label': 'tab:minutes', 'csv': 'Table_A8.csv'}, 'Table_A9': {'title': 'Full text ECB introductions and market changes.', 'label': 'tab:ecbmarket', 'csv': 'Table_A9.csv'}, 'Table_A10': {'title': 'Annual leader-follower associations.', 'label': 'tab:annual-leaders', 'csv': 'Table_A10.csv'}, 'Table_A11': {'title': 'Annual macroeconomic associations.', 'label': 'tab:macro', 'csv': 'Table_A11.csv'}, 'Table_A12': {'title': 'How consistent repeated ratings are, by concept.', 'label': 'tab:reliability', 'csv': 'Table_A12.csv'}, 'Table_A13': {'title': 'ECB results using the fuller introductory-statement archive.', 'label': 'tab:ecbrepair', 'csv': 'Table_A13.csv'}, 'Table_A14': {'title': 'Testing whether leader-measurement methods actually differ.', 'label': 'tab:coefficientdifferences', 'csv': 'Table_A14.csv'}, 'Table_A15': {'title': 'How well the reconstruction recovers the original labels.', 'label': 'tab:reconstruction', 'csv': 'Table_A15.csv'}, 'Table_A16': {'title': 'Directional-claim alignment across windows and draws.', 'label': 'tab:claims', 'csv': 'Table_A16.csv'}, 'Table_A17': {'title': 'Hawkish results under different clustering choices.', 'label': 'tab:clustering', 'csv': 'Table_A17.csv'}, 'Table_A18': {'title': 'A worked example with common news and zero transmission.', 'label': 'tab:counterexample', 'csv': 'Table_A18.csv'}, 'Table_A19': {'title': 'How precisely each estimate is pinned down.', 'label': 'tab:precision', 'csv': 'Table_A19.csv'}, 'Table_A20': {'title': 'Association with the preceding Fed change alone.', 'label': 'tab:preceding', 'csv': 'Table_A20.csv'}}
COMPUTATION_KEYS = {'Table_1': 'tab:methods', 'Table_2': 'tab:sensitivity', 'Table_3': 'tab:phrases', 'Table_A1': 'tab:coverage-2', 'Table_A2': 'tab:measure', 'Table_A3': 'tab:sync', 'Table_A4': 'tab:balanced', 'Table_A5': 'tab:spillover', 'Table_A6': 'tab:placebo', 'Table_A7': 'tab:contrasts', 'Table_A8': 'tab:minutes', 'Table_A9': 'tab:ecbmarket', 'Table_A10': 'tab:annual-leaders', 'Table_A11': 'tab:macro', 'Table_A12': 'tab:reliability', 'Table_A13': 'tab:ecbrepair', 'Table_A14': 'tab:coefficientdifferences', 'Table_A15': 'tab:reconstruction', 'Table_A16': 'tab:claims', 'Table_A17': 'tab:clustering', 'Table_A18': 'tab:counterexample', 'Table_A19': 'tab:precision', 'Table_A20': 'tab:preceding'}

TABLE_TEX={}
PUBLISH_BY_LABEL={}


### Figures

In [ ]:
%%writefile code/figures.py
"""V9 publication figures. Visual design only; every plotted estimate and interval is preserved."""
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator, FuncFormatter
import numpy as np
import pandas as pd
from data import ROOT, CONCEPTS, read_jsonl
from exhibits import read

NAVY = '#263F55'
RUST = '#A85B3F'
SLATE = '#7C858A'
INK = '#252B30'
RULE = '#C7CDD0'
GRID = '#E7EAEC'
PALE = '#F3EEE8'
LABELS = {
    'activity_outlook': 'Activity outlook',
    'inflation_outlook': 'Inflation outlook',
    'hawkish_reaction': 'Hawkish language',
    'supply_narrative': 'Supply attribution',
    'uncertainty': 'Uncertainty',
    'fx_pressure': 'Exchange rate pressure',
    'financial_stability': 'Financial stability',
}
plt.rcParams.update({
    'font.family': 'serif', 'font.serif': ['DejaVu Serif'],
    'mathtext.fontset': 'dejavuserif',
    'font.size': 10, 'axes.labelsize': 10, 'axes.titlesize': 10,
    'xtick.labelsize': 9, 'ytick.labelsize': 9, 'legend.fontsize': 9,
    'text.color': INK, 'axes.labelcolor': INK,
    'axes.edgecolor': RULE, 'axes.linewidth': .65,
    'axes.spines.top': False, 'axes.spines.right': False,
    'xtick.color': INK, 'ytick.color': INK,
    'xtick.major.size': 3, 'ytick.major.size': 3,
    'xtick.major.width': .6, 'ytick.major.width': .6,
    'xtick.major.pad': 5, 'ytick.major.pad': 5,
    'axes.grid': False, 'axes.axisbelow': True,
    'legend.frameon': False, 'legend.handletextpad': .65,
    'legend.borderaxespad': 0, 'legend.labelspacing': .65,
    'pdf.fonttype': 42, 'ps.fonttype': 42,
    'savefig.facecolor': 'white', 'figure.facecolor': 'white',
    'lines.solid_capstyle': 'round',
})
FIGURE_KEYS = [
    'figure_annual', 'timeline', 'figure_phrases',
    'figure_coverage_singlepanel', 'figure_attention', 'figure_phrases_appendix',
]
HORIZONS = [7, 14, 21, 30, 45]


def finish(fig, key):
    (ROOT / 'figures').mkdir(parents=True, exist_ok=True)
    fig.savefig(ROOT / 'figures' / f'{key}.pdf', bbox_inches='tight', pad_inches=.07)
    fig.savefig(ROOT / 'figures' / f'{key}.png', dpi=220, bbox_inches='tight', pad_inches=.07)
    plt.close(fig)


def tidy(ax, grid='y'):
    ax.spines['left'].set_visible(False)
    ax.tick_params(axis='y', length=0)
    if grid:
        ax.grid(axis=grid, color=GRID, linewidth=.55)


def number(value, position=None):
    if abs(value) < 1e-10:
        return '0'
    return f'{value:.2f}'.rstrip('0').rstrip('.').replace('-', '\N{MINUS SIGN}')


def zero(ax):
    ax.axhline(0, color=INK, linewidth=.8, zorder=1)


def series_key(color, marker='o', filled=True):
    return Line2D([], [], color=color, marker=marker, linestyle='none', markersize=5,
                  markerfacecolor=color if filled else 'white', markeredgewidth=1.0)


def intervals(ax, g, color, marker, offset, filled=True):
    """Plot exactly the confidence limits saved by the analysis; no inference is recomputed."""
    assert g.horizon.tolist() == HORIZONS
    assert np.isfinite(g[['estimate', 'ci_low', 'ci_high']].to_numpy()).all()
    assert (g.ci_low <= g.estimate).all() and (g.ci_high >= g.estimate).all()
    ax.errorbar(g.horizon.to_numpy() + offset, g.estimate.to_numpy(),
                yerr=np.vstack([g.estimate - g.ci_low, g.ci_high - g.estimate]),
                fmt=marker, linestyle='none', color=color,
                markerfacecolor=color if filled else 'white', markeredgewidth=1.0,
                markersize=4.8, elinewidth=.95, capsize=2.6, capthick=.85, zorder=3)


def make_annual():
    annual, exclusions = read('annual_balanced'), read('annual_comovement')
    order = ['hawkish_reaction', 'inflation_outlook', 'supply_narrative', 'fx_pressure',
             'uncertainty', 'financial_stability', 'activity_outlook']
    samples = [
        (annual, 'All', NAVY, 'o', True, -.21, 'All jurisdictions'),
        (annual, 'Balanced', RUST, 's', False, 0, 'Balanced, 2005–24'),
        (exclusions, 'Excl. US/ECB', SLATE, 'D', True, .21, 'Excluding Fed and ECB'),
    ]
    fig, ax = plt.subplots(figsize=(6.9, 4.1))
    fig.subplots_adjust(left=.285, right=.98, bottom=.16, top=.85)
    y = np.arange(len(order))
    for yy in y:
        ax.hlines(yy, 0, 42, color=GRID, linewidth=.55, zorder=0)
    handles, legends = [], []
    for source, sample, color, marker, filled, offset, label in samples:
        g = source[source['sample'].eq(sample)].set_index('concept')
        assert g.index.is_unique and set(order) <= set(g.index)
        values = g.reindex(order).projection_share.to_numpy()
        assert np.isfinite(values).all() and ((values >= 0) & (values <= 100)).all()
        ax.scatter(values, y + offset, marker=marker, s=29,
                   facecolors=color if filled else 'white', edgecolors=color,
                   linewidths=1.05, zorder=3)
        handles.append(series_key(color, marker, filled)); legends.append(label)
    ax.set(yticks=y, yticklabels=[LABELS[c] for c in order],
           xlim=(0, 42), ylim=(len(order) - .5, -.5),
           xlabel='Annual projection share (%)', xticks=[0, 10, 20, 30, 40])
    ax.tick_params(axis='y', length=0, pad=10, labelsize=9.4)
    ax.spines['left'].set_visible(False)
    fig.legend(handles, legends, ncol=3, loc='upper center', bbox_to_anchor=(.53, .995),
               columnspacing=1.35, handlelength=1, fontsize=8.8)
    finish(fig, 'figure_annual')


def make_timeline():
    fig, ax = plt.subplots(figsize=(7.05, 2.65))
    fig.subplots_adjust(left=.02, right=.98, bottom=.02, top=.98)
    prior, preceding, foreign, following = .1, 2.05, 3.5, 5.1
    ax.set(xlim=(-.65, 5.8), ylim=(-1.02, 1.74))
    ax.annotate('', xy=(5.66, 0), xytext=(-.44, 0),
                arrowprops={'arrowstyle': '-|>', 'lw': .85, 'color': SLATE, 'mutation_scale': 8})
    # Each Fed change ends at its own release and begins at the preceding Fed release.
    for left, right, y, label in [
        (prior, preceding, .85, 'Preceding Fed change  $\\Delta q_F^-$'),
        (preceding, following, 1.43, 'Following Fed change  $\\Delta q_F^+$'),
    ]:
        ax.vlines([left, right], .06, y, color=RULE, lw=.65, linestyle=(0, (2, 3)))
        ax.annotate('', xy=(right, y), xytext=(left, y),
                    arrowprops={'arrowstyle': '-|>', 'color': NAVY, 'lw': 1.0, 'mutation_scale': 9})
        ax.text((left + right) / 2, y + .12, label, ha='center', va='bottom', fontsize=9.2, color=NAVY)
    for x in [prior, preceding, following]:
        ax.plot(x, 0, marker='o', ms=5.8, mfc='white', mec=NAVY, mew=1.15, zorder=4)
    ax.plot(foreign, 0, marker='D', ms=5.7, mfc=RUST, mec=RUST, zorder=4)
    for x, label in [
        (prior, 'Earlier Fed\nrelease'),
        (preceding, 'Preceding Fed\n$\\tau_F^-(t)$'),
        (following, 'Following Fed\n$\\tau_F^+(t)$'),
    ]:
        ax.text(x, -.16, label, ha='center', va='top', fontsize=9.2, linespacing=1.45)
    ax.text(foreign, .20, 'Foreign statement\nlevel $z_{it}$ at $t$',
            ha='center', va='bottom', fontsize=9.4, color=RUST, linespacing=1.45)
    for left, right in [(preceding, foreign), (foreign, following)]:
        ax.annotate('', xy=(right, -.68), xytext=(left, -.68),
                    arrowprops={'arrowstyle': '|-|', 'color': SLATE, 'lw': .65, 'mutation_scale': 4})
        ax.text((left + right) / 2, -.80, '1–45 days', ha='center', va='top', fontsize=8.8, color=INK)
    ax.axis('off')
    finish(fig, 'timeline')


def make_phrases():
    d = read('phrase_adoption')
    d = d[d.design.eq('past_window') & d['split'].eq('all')]
    fig, ax = plt.subplots(figsize=(6.8, 3.15))
    fig.subplots_adjust(left=.13, right=.98, bottom=.22, top=.83)
    tidy(ax)
    for variant, color, marker, offset, filled in [
        ('all_novel', NAVY, 'o', -.65, True),
        ('meaningful_novel', RUST, 's', .65, False),
    ]:
        g = d[d.variant.eq(variant)].sort_values('horizon')
        assert g.ci_low.ge(-.10).all() and g.ci_high.le(.15).all()
        intervals(ax, g, color, marker, offset, filled)
    zero(ax)
    ax.set(xlim=(3.5, 48.5), ylim=(-.10, .15), xticks=HORIZONS,
           yticks=[-.10, -.05, 0, .05, .10, .15],
           xlabel='Preceding window (days)', ylabel='Preceding minus following (pp)')
    ax.yaxis.set_major_formatter(FuncFormatter(number))
    fig.legend([series_key(NAVY), series_key(RUST, 's', False)],
               ['All new phrases', 'Substantive phrases'],
               ncol=2, loc='upper center', bbox_to_anchor=(.54, .98),
               columnspacing=2.2, handlelength=1.0)
    finish(fig, 'figure_phrases')


def make_coverage():
    d = pd.DataFrame(read_jsonl(ROOT / 'derived/corpus.jsonl'))
    d['year'] = pd.to_datetime(d.date).dt.year
    a = d.groupby('year').agg(texts=('doc_id', 'size'), countries=('country', 'nunique'))
    fig, axes = plt.subplots(2, 1, figsize=(6.8, 4.4), sharex=True,
                             gridspec_kw={'height_ratios': [1.15, 1]})
    fig.subplots_adjust(left=.095, right=.98, top=.92, bottom=.24, hspace=.40)
    top, bottom = axes
    top.bar(a.index, a.texts, width=.72, color=NAVY, zorder=3)
    top.set_ylim(0, max(a.texts) * 1.12)
    top.yaxis.set_major_locator(MaxNLocator(4, integer=True))
    bottom.plot(a.index, a.countries, color=RUST, lw=1.45, marker='o',
                ms=2.4, mfc=RUST, mec=RUST, zorder=3)
    bottom.set(ylim=(0, 55), yticks=[0, 25, 50],
               xlim=(1989.25, 2024.75), xlabel='Year',
               xticks=[1990, 1995, 2000, 2005, 2010, 2015, 2020, 2024])
    for ax, name in zip(axes, ['(a) Statements', '(b) Jurisdictions']):
        tidy(ax)
        ax.set_title(name, loc='left', pad=8, fontsize=10)
    fig.text(.095, .075, 'Median words per statement', fontsize=8.8, color=INK, va='center')
    positions = [.49, .65, .80, .945]
    for x, (lo, hi) in zip(positions, [(1990, 1999), (2000, 2009), (2010, 2019), (2020, 2024)]):
        median = d.loc[d.year.between(lo, hi), 'word_count'].median()
        fig.text(x, .096, f'{lo}–{str(hi)[-2:]}', fontsize=8.1, color=SLATE, ha='center')
        fig.text(x, .05, f'{median:,.0f}', fontsize=9.2, color=INK, ha='center')
    finish(fig, 'figure_coverage_singlepanel')


def make_attention():
    full, balanced = read('attention_annual_equal_country'), read('attention_balanced_2005_2024')
    concepts = ['supply_narrative', 'financial_stability', 'fx_pressure']
    fig, axes = plt.subplots(3, 1, figsize=(6.8, 5.0), sharex=True, sharey=True)
    fig.subplots_adjust(left=.11, right=.98, bottom=.115, top=.86, hspace=.54)
    for j, (ax, concept) in enumerate(zip(axes, concepts)):
        tidy(ax)
        if concept == 'supply_narrative':
            ax.axvspan(2021, 2022, color=PALE, lw=0, zorder=0)
        if concept == 'financial_stability':
            ax.axvspan(2008, 2009, color=PALE, lw=0, zorder=0)
            ax.axvline(2023, color=RUST, lw=.7, alpha=.7, ls=(0, (2, 3)), zorder=1)
        for source, color, ls, width in [
            (full, NAVY, '-', 1.25), (balanced, RUST, (0, (4, 2)), 1.45),
        ]:
            g = source[source.concept.eq(concept)].sort_values('year')
            assert g.year.is_unique and np.isfinite(g.mean_coverage).all()
            assert g.mean_coverage.between(0, 12).all()
            ax.plot(g.year, g.mean_coverage, color=color, linestyle=ls, lw=width, zorder=3)
        ax.set_title(f'({chr(97 + j)}) {LABELS[concept]}', loc='left', pad=6, fontsize=10)
        ax.set(ylim=(0, 12), yticks=[0, 5, 10], xlim=(1989.3, 2024.7))
    axes[-1].set(xlabel='Year', xticks=[1990, 2000, 2010, 2020, 2024])
    fig.supylabel('Coverage of substantive clauses (%)', x=.015, fontsize=10)
    handles = [Line2D([], [], color=NAVY, lw=1.25),
               Line2D([], [], color=RUST, lw=1.45, linestyle=(0, (4, 2)))]
    fig.legend(handles, ['All available jurisdictions', 'Fixed 21 jurisdictions, 2005–24'],
               ncol=2, loc='upper center', bbox_to_anchor=(.53, .985),
               handlelength=2.0, columnspacing=1.7, fontsize=8.9)
    finish(fig, 'figure_attention')


def make_phrases_appendix():
    d = read('phrase_adoption')
    d = d[d.design.eq('past_window')]
    fig, axes = plt.subplots(2, 2, figsize=(7.15, 6.7), sharex=True, sharey=True)
    fig.subplots_adjust(left=.105, right=.985, bottom=.20, top=.935, hspace=.86, wspace=.20)
    labels = {'all': 'All statements', 'names_Fed': 'Names the Fed',
              'does_not_name_Fed': 'Does not name the Fed',
              'Official_English': 'Official English', 'Database_translated': 'Translated'}
    style = [(NAVY, 'o', True), (RUST, 's', False), (SLATE, '^', True)]
    for row, variant in enumerate(['all_novel', 'meaningful_novel']):
        for col, splits in enumerate([['all', 'names_Fed', 'does_not_name_Fed'],
                                      ['all', 'Official_English', 'Database_translated']]):
            ax = axes[row, col]
            tidy(ax)
            handles, names = [], []
            for j, (split, (color, marker, filled)) in enumerate(zip(splits, style)):
                g = d[d.variant.eq(variant) & d['split'].eq(split)].sort_values('horizon')
                assert g.ci_low.ge(-.4).all() and g.ci_high.le(.8).all()
                intervals(ax, g, color, marker, (j - 1) * 1.0, filled)
                handles.append(series_key(color, marker, filled))
                names.append(f'{labels[split]}  ($n$ = {int(g.n.min()):,}–{int(g.n.max()):,})')
            zero(ax)
            ax.set(xlim=(3.5, 48.5), ylim=(-.4, .8), xticks=HORIZONS, yticks=[-.4, 0, .4, .8])
            ax.yaxis.set_major_formatter(FuncFormatter(number))
            ax.tick_params(labelbottom=True, labelleft=True, labelsize=8.5)
            variant_name = 'All new phrases' if row == 0 else 'Substantive phrases'
            group_name = 'Fed mention' if col == 0 else 'source language'
            ax.set_title(f'({chr(97 + 2 * row + col)}) {variant_name}\n{group_name}',
                         loc='left', pad=9, fontsize=9.5, linespacing=1.3)
            ax.legend(handles, names, loc='upper left', bbox_to_anchor=(-.01, -.20),
                      fontsize=7.5, handlelength=1.0, handletextpad=.5, labelspacing=.4)
    fig.supxlabel('Preceding window (days)', y=.025, fontsize=10)
    fig.supylabel('Preceding minus following (pp)', x=.008, fontsize=10)
    finish(fig, 'figure_phrases_appendix')


def make_figure(key):
    if key != 'timeline' and not key.startswith('figure_'):
        key = 'figure_' + key
    makers = {
        'figure_annual': make_annual,
        'timeline': make_timeline,
        'figure_phrases': make_phrases,
        'figure_coverage_singlepanel': make_coverage,
        'figure_attention': make_attention,
        'figure_phrases_appendix': make_phrases_appendix,
    }
    if key not in makers:
        raise KeyError(key)
    makers[key]()
    return ROOT / 'figures' / f'{key}.png'


def all_figures():
    for key in FIGURE_KEYS:
        make_figure(key)
    print('Regenerated', len(FIGURE_KEYS), 'vector figures and PNG previews.')


if __name__ == '__main__':
    all_figures()


### Publication

In [ ]:
%%writefile code/publication.py
"""Regenerate exhibits and verify the authoritative manuscript files without rewriting them."""
from pathlib import Path
import re,json,hashlib,shutil,html
import pandas as pd
from IPython.display import display,HTML,Image
from data import ROOT,read_jsonl
import exhibits
from figures import make_figure
PACKAGE=None
SOURCES={};TABLES={};FIGURES={};SOURCE_HASHES={};EXHIBIT_COUNTS={}
GENERATED_TABLES=set();GENERATED_FIGURES=set();TABLE_DATA={}
# These are the retained V9 exhibits; an accidental source omission must fail.
EXPECTED_EXHIBITS={'main.tex': {'table': ['tab:methods'], 'figure': ['fig:annual', 'fig:timing', 'fig:phrase-profile']}, 'appendix.tex': {'table': ['tab:coverage-2', 'tab:sync', 'tab:placebo', 'tab:sensitivity', 'tab:clustering', 'tab:counterexample', 'tab:claims', 'tab:reliability', 'tab:coefficientdifferences', 'tab:reconstruction'], 'figure': ['fig:coverage', 'fig:attention', 'fig:phrase-subgroups']}}
def braced(text,start):
    assert text[start]=='{'
    depth=1;i=start+1
    while depth:
        if text[i]=='{' and text[i-1]!='\\':depth+=1
        elif text[i]=='}' and text[i-1]!='\\':depth-=1
        i+=1
    return text[start+1:i-1]
def configure(package):
    global PACKAGE
    PACKAGE=Path(package)
    for collection in [SOURCES,TABLES,FIGURES,SOURCE_HASHES,EXHIBIT_COUNTS,TABLE_DATA]:collection.clear()
    GENERATED_TABLES.clear();GENERATED_FIGURES.clear()
    for document in ['main.tex','appendix.tex']:
        source=(PACKAGE/document).read_text();SOURCES[document]=source
        SOURCE_HASHES[document]=hashlib.sha256((PACKAGE/document).read_bytes()).hexdigest()
        counts={'table':0,'figure':0}
        for match in re.finditer(r'\\begin\{(table|figure)\}.*?\\end\{\1\}',source,re.S):
            kind=match[1];block=match[0];label=re.search(r'\\label\{([^}]+)\}',block)[1]
            counts[kind]+=1;number=('A' if document=='appendix.tex' else '')+str(counts[kind])
            start=block.index('\\caption{')+len('\\caption')
            info={'document':document,'number':number,'title':braced(block,start),'block':block}
            if kind=='table':
                key=next(k for k,v in exhibits.TABLES.items() if v['label']==label)
                info['key']=key;TABLES[label]=info
                exhibits.TABLES[key].update(template=block,title=info['title'],document=document)
                exhibits.PUBLISH_BY_LABEL[label]='Table_'+number
            else:
                info['file']=re.search(r'\\includegraphics(?:\[[^]]*\])?\{([^}]+)\}',block)[1]
                FIGURES[label]=info
        EXHIBIT_COUNTS[document]=counts
    assert len(TABLES)==sum(v['table'] for v in EXHIBIT_COUNTS.values()),'Duplicate table labels'
    assert len(FIGURES)==sum(v['figure'] for v in EXHIBIT_COUNTS.values()),'Duplicate figure labels'
    for document,expected in EXPECTED_EXHIBITS.items():
        for kind,collection in [('table',TABLES),('figure',FIGURES)]:
            actual={label for label,info in collection.items() if info['document']==document}
            assert actual==set(expected[kind]),(document,kind,'V9 exhibit inventory differs',sorted(actual),expected[kind])

def numerical_values():
    raw=read_jsonl(ROOT/'derived/corpus.jsonl')
    main=pd.read_csv(ROOT/'results/main_results.csv')
    common=pd.read_csv(ROOT/'results/method_common_sample_results.csv')
    annual=pd.read_csv(ROOT/'results/annual_balanced.csv')
    def effect(method,concept,column='estimate',common_sample=False):
        d=common if common_sample else main[(main.spec=='joint')&(main.leg=='difference')]
        return float(d[(d.method==method)&(d.concept==concept)][column].iloc[0])
    def share(sample,concept):return float(annual[(annual['sample']==sample)&(annual.concept==concept)].projection_share.iloc[0])
    values={'NStatements':f'{len(raw):,}','NCountries':str(len({r['country'] for r in raw})),
      'HawkShareAll':f"{share('All','hawkish_reaction'):.1f}",
      'InflationShareAll':f"{share('All','inflation_outlook'):.1f}",
      'HawkShareBalanced':f"{share('Balanced','hawkish_reaction'):.1f}",
      'InflationShareBalanced':f"{share('Balanced','inflation_outlook'):.1f}"}
    specs={'LLMInflationDiff':('Continuous LLM','inflation_outlook','estimate',False),
      'LLMInflationSE':('Continuous LLM','inflation_outlook','std_error',False),
      'LLMInflationQ':('Continuous LLM','inflation_outlook','q_value',False),
      'PairInflationDiff':('Pairwise leader / lexical outcome','inflation_outlook','estimate',False),
      'EmbHawkDiff':('Full embeddings','hawkish_reaction','estimate',False),
      'EmbHawkCommonDiff':('Full embeddings','hawkish_reaction','estimate',True),
      'EmbHawkCommonSE':('Full embeddings','hawkish_reaction','std_error',True),
      'EmbHawkCommonQ':('Full embeddings','hawkish_reaction','q_value',True)}
    values.update({name:f'{effect(*args):.3f}' for name,args in specs.items()})
    # Additional calculated values used by appendix checks.
    aliases={'CommonEmbHawkDiff':'EmbHawkCommonDiff','CommonEmbHawkSE':'EmbHawkCommonSE','CommonEmbHawkQ':'EmbHawkCommonQ',
             'HawkishYearShare':'HawkShareAll','InflationYearShare':'InflationShareAll'}
    values.update({key:values[value] for key,value in aliases.items()})
    lexical_joint=main[(main.method=='TF-IDF')&(main.spec=='joint')&(main.leg=='difference')]
    assert lexical_joint.n.nunique()==1 and lexical_joint.n_events.nunique()==1
    values['NPairedStatements']=f"{int(lexical_joint.n.iloc[0]):,}"
    values['NFedEvents']=str(int(lexical_joint.n_events.iloc[0]))
    sensitivity=pd.read_csv(ROOT/'results/joint_sensitivity.csv')
    for name,spec in [('LexInflationOwnLag','own_previous_statement'),('LexInflationEqualEvents','equal_events')]:
        row=sensitivity[(sensitivity.concept=='inflation_outlook')&(sensitivity.spec==spec)]
        assert len(row)==1,(name,'Expected one sensitivity estimate')
        values[name]=f"{float(row.estimate.iloc[0]):.3f}"
    # Values cited directly in the main text
    values.update({'ShareHawkAll':values['HawkShareAll'],'ShareInflAll':values['InflationShareAll'],
      'ShareHawkBal':values['HawkShareBalanced'],'ShareInflBal':values['InflationShareBalanced'],
      'ShareSupplyAll':f"{share('All','supply_narrative'):.1f}",
      'DHawkEmb':values['EmbHawkDiff'],
      'qHawkEmb':f"{effect('Full embeddings','hawkish_reaction','q_value'):.3f}"})
    reliability=pd.read_csv(ROOT/'results/test_retest_reliability.csv')
    for name,concept in [('RelActivity','activity_outlook'),('RelHawk','hawkish_reaction'),('RelSupply','supply_narrative'),
                         ('RelFinStab','financial_stability'),('RelFX','fx_pressure')]:
        row=reliability[(reliability.kind=='level')&(reliability.country=='ALL')&(reliability.concept==concept)]
        assert len(row)>=1,(name,'Missing reliability row')
        values[name]=f"{float(row.correlation.iloc[0]):.2f}"
    fragile=sensitivity[(sensitivity.concept=='inflation_outlook')&sensitivity.spec.isin(['drop_2020_2022','own_previous_statement','equal_events'])]
    assert len(fragile)==3,'Expected three inflation sensitivity estimates'
    values['SensInflMax']=f"{float(fragile.estimate.max()):.3f}"
    phrases=pd.read_csv(ROOT/'results/phrase_adoption.csv')
    row=phrases[(phrases.variant=='all_novel')&(phrases['split']=='all')&(phrases.horizon==45)&(phrases.design=='past_window')]
    assert len(row)==1,'Expected one all-novel phrase contrast'
    values['PhraseContrast']=f"{float(row.estimate.iloc[0]):.3f}"
    values['PhraseSE']=f"{float(row.std_error.iloc[0]):.3f}"
    window=phrases[(phrases.variant=='all_novel')&(phrases['split']=='all')&(phrases.design=='past_window')&phrases.horizon.isin([7,14,21,30,45])]
    assert len(window)==5,'Expected the five reported phrase horizons'
    values['PhraseMin']=f"{float(window.estimate.min()):.3f}"
    values['PhraseMax']=f"{float(window.estimate.max()):.3f}"
    return values
def verify_text_values():
    """Check the literal numerical claims against freshly calculated estimates."""
    calculated=numerical_values();checked={}
    patterns=[
      ('main.tex',r'Scoring ([\d,]+) monetary policy statements from (\d+) jurisdictions', ['NStatements','NCountries']),
      ('main.tex',r'The sample contains ([\d,]+) monetary policy statements from (\d+) jurisdictions', ['NStatements','NCountries']),
      ('main.tex',r'largest projection share \(([\d.]+) percent\), followed by the inflation outlook \(([\d.]+)\) and supply attribution \(([\d.]+)\)', ['ShareHawkAll','ShareInflAll','ShareSupplyAll']),
      ('main.tex',r'whose shares are higher \(([\d.]+) and ([\d.]+) percent\)', ['ShareHawkBal','ShareInflBal']),
      ('main.tex',r'gives a clear contrast \(([\d.]+), \$q=([\d.]+)\$\)', ['DHawkEmb','qHawkEmb']),
      ('main.tex',r'activity outlook \(\$r=([\d.]+)\$\) and hawkish language \(\$r=([\d.]+)\$\)', ['RelActivity','RelHawk']),
      ('main.tex',r'less so for supply attribution \(\$r=([\d.]+)\$\), financial stability \(\$r=([\d.]+)\$\), and exchange rate pressure \(\$r=([\d.]+)\$\)', ['RelSupply','RelFinStab','RelFX']),
      ('main.tex',r'difference ranges from ([\d.]+) to ([\d.]+) percentage points', ['PhraseMin','PhraseMax']),
      ('main.tex',r'falling to about ([\d.]+) or below', ['SensInflMax']),
      ('main.tex',r'direct LLM measure, \$D=([\d.]+)\$ \(\$q=([\d.]+)\$\)', ['LLMInflationDiff','LLMInflationQ']),
      ('appendix.tex',r'contains ([\d,]+) statements from (\d+) jurisdictions', ['NStatements','NCountries']),
      ('appendix.tex',r'embedding contrast falls from ([\d.]+) to ([\d.]+) \(standard error ([\d.]+), BH \$q=([\d.]+)\$\)', ['EmbHawkDiff','EmbHawkCommonDiff','EmbHawkCommonSE','EmbHawkCommonQ']),
    ]
    for i,(filename,pattern,names) in enumerate(patterns):
        matches=re.findall(pattern,SOURCES[filename])
        assert len(matches)==1,(filename,'Numerical sentence missing or ambiguous',pattern,matches)
        found=matches[0] if isinstance(matches[0],tuple) else (matches[0],)
        assert len(found)==len(names)
        for name,value in zip(names,found):
            assert value==calculated[name],(filename,name,value,calculated[name])
            checked[f'{filename}:{i}:{name}']=value
    common=pd.read_csv(ROOT/'results/method_common_sample_results.csv')
    row=common[(common.method=='Full embeddings')&(common.concept=='hawkish_reaction')]
    assert len(row)==1
    match=re.search(r'On ([\d,]+) hawkish observations eligible under all four methods',SOURCES['appendix.tex'])
    assert match and match[1]==f"{int(row.n.iloc[0]):,}",'Common-sample size mismatch'
    checked['appendix.tex:common_sample_n']=match[1]
    for filename,source in SOURCES.items():
        assert not re.search(r'\\newcommand\{\\[^}]+\}\{[-+]?\d[\d,.]*\}',source),(filename,'Numerical macros must remain expanded')
    return checked

def verify_appendix_references():
    """Audit the main paper's literal cross-document references without auxiliary files."""
    appendix=SOURCES['appendix.tex'];main=SOURCES['main.tex']
    section=0;subsection=0;locations={}
    for match in re.finditer(r'\\(section|subsection|label)\{([^}]+)\}',appendix):
        kind,value=match.groups()
        if kind=='section':section+=1;subsection=0
        elif kind=='subsection':subsection+=1
        else:locations[value]=chr(64+section)+(f'.{subsection}' if subsection else '')
    checks=[('Online Appendix~A.1','app:sample','A.1'),
      ('Online Appendix~A.2','app:ratings','A.2'),
      ('Appendix~F','app:exemplars','F'),
      ('Online Appendix~E','app:diagnostics','E'),
      ('Online Appendix~B','app:annual','B'),
      ('Online Appendices~C and~E','app:sensitivity','C'),
      ('Online Appendices~C and~E','app:diagnostics','E'),
      ('Online Appendix~D','app:wording','D')]
    for fragment,label,number in checks:
        assert fragment in main,(fragment,'Main appendix reference missing')
        assert locations[label]==number,(label,locations[label],number)
    number=TABLES['tab:counterexample']['number']
    assert f'Online Appendix Table~{number}' in main,'Common-news table reference is stale'
    prompts=re.findall(r'\\captionof\{prompt\}\{[^\n]+\}\\label\{([^}]+)\}',appendix)
    expected=['prompt:rating-system','prompt:direct','prompt:pairwise','prompt:phrase-system','prompt:phrase-classification','prompt:reconstruction']
    assert prompts==expected,('Prompt inventory/order changed',prompts)
    for label in prompts:assert r'\ref{'+label+'}' in appendix,('Prompt is not cited in prose',label)
    result={'section_references_verified':len(checks),'counterexample_table':number,
            'prompt_captions_verified':len(prompts),'prompt_labels':prompts,
            'main_phrase_figure':FIGURES['fig:phrase-profile']['file'],
            'appendix_subgroup_figure':FIGURES['fig:phrase-subgroups']['file']}
    (ROOT/'audit/manuscript_checks.json').write_text(json.dumps({'text_values':verify_text_values(),'appendix_references':result},indent=2)+'\n')
    return result

def body(block):
    return re.search(r'\\midrule(.*?)\\bottomrule',block,re.S)[1]
def normalized(block):
    value=body(block)
    value=re.sub(r'(?<!\\)%[^\n]*','',value)
    value=re.sub(r'\\addlinespace(?:\[[^]]*\])?','',value)
    return re.sub(r'\s+','',value)
def split_rows(value):
    rows=[];depth=0;start=0;i=0
    while i<len(value):
        if value[i]=='{' and (i==0 or value[i-1]!='\\'):depth+=1
        elif value[i]=='}' and (i==0 or value[i-1]!='\\'):depth-=1
        elif depth==0 and value[i:i+2]=='\\\\':
            rows.append(value[start:i]);i+=2;start=i;continue
        i+=1
    if value[start:].strip():rows.append(value[start:])
    return rows

def cell_text(value):
    value=re.sub(r'\\(?:addlinespace)(?:\[[^]]*\])?','',value)
    value=value.replace('\\\\','\n').replace(r'$-$','−')
    for old,new in [(r'\%', '%'),(r'\rho','ρ'),(r'\pm','±'),(r'\&','&'),('~',' ')]:value=value.replace(old,new)
    value=re.sub(r'\\shortstack(?:\[[^]]*\])?','',value)
    value=re.sub(r'\\[A-Za-z]+(?:\[[^]]*\])?','',value)
    value=value.replace('{','').replace('}','').replace('$','')
    return html.escape(value.strip()).replace('\n','<br>')
def table_html(block):
    content=re.search(r'\\toprule(.*?)\\bottomrule',block,re.S)[1]
    parts=content.split('\\midrule',1);header=split_rows(parts[0]);rows=split_rows(parts[1]);out=[]
    for is_header,collection in [(True,header),(False,rows)]:
        for row in collection:
            row=re.sub(r'\\addlinespace(?:\[[^]]*\])?','',row).strip()
            if not row:continue
            span=re.match(r'\\multicolumn\{(\d+)\}\{[^}]*\}\{',row)
            if span:
                label=braced(row,span.end()-1)
                out.append('<tr><td colspan="'+span[1]+'" style="text-align:left;padding-top:12px"><em>'+cell_text(label)+'</em></td></tr>');continue
            cells=re.split(r'(?<!\\)&',row)
            tag='th' if is_header else 'td'
            out.append('<tr>'+''.join('<'+tag+' style="padding:4px 10px;text-align:'+('left' if i==0 else 'right')+'">'+cell_text(c)+'</'+tag+'>' for i,c in enumerate(cells))+'</tr>')
    return '<div style="overflow-x:auto"><table style="border-collapse:collapse;font-family:serif;border-top:1.5px solid #333;border-bottom:1.5px solid #333">'+''.join(out)+'</table></div>'

def show_table(label):
    info=TABLES[label];frame=exhibits.make_table(info['key'])
    regenerated=exhibits.TABLE_TEX[label]
    assert normalized(regenerated)==normalized(info['block']),(label,'Fresh table values differ from manuscript. Sources were not modified.')
    TABLE_DATA[label]=frame;GENERATED_TABLES.add(label)
    display(HTML(table_html(regenerated)))
    #print('Verified against '+info['document']+'; numerical source: TABLE_DATA['+repr(label)+'].')

def show_figure(label):
    info=FIGURES[label];filename=Path(info['file']).name;key=Path(filename).stem
    make_figure(key)
    destination=PACKAGE/info['file'];destination.parent.mkdir(parents=True,exist_ok=True)
    shutil.copy2(ROOT/'figures'/filename,destination)
    display(Image(filename=str(ROOT/'figures'/f'{key}.png'),width=950))
    GENERATED_FIGURES.add(label)
    print('Saved '+info['file'])

def verify_all():
    assert GENERATED_TABLES==set(TABLES)
    assert GENERATED_FIGURES==set(FIGURES)
    checked=verify_text_values()
    cross_references=verify_appendix_references()
    for filename,expected in SOURCE_HASHES.items():
        assert hashlib.sha256((PACKAGE/filename).read_bytes()).hexdigest()==expected,(filename,'Source changed')
        source=SOURCES[filename]
        labels=re.findall(r'\\label\{([^}]+)\}',source)
        assert len(labels)==len(set(labels)),(filename,'Duplicate labels')
        refs=set(re.findall(r'\\(?:eqref|ref)\{([^}]+)\}',source))
        assert not refs-set(labels),(filename,'Unresolved references',refs-set(labels))
        for name in re.findall(r'\\includegraphics(?:\[[^]]*\])?\{([^}]+)\}',source):assert (PACKAGE/name).is_file(),name
    verification=json.loads((ROOT/'audit/verification.json').read_text())
    assert verification['status']=='passed'
    return {'status':'passed','tables_verified':len(GENERATED_TABLES),'figures_generated':len(GENERATED_FIGURES),'numerical_text_values_verified':len(checked),'appendix_references':cross_references,'exhibits_by_document':EXHIBIT_COUNTS,'manuscript_files_unchanged':True,'independent_statistical_checks':'passed'}


## Recompute the results

### Benchmark and grid diagnostics

In [ ]:
import diagnostics
with np.errstate(over="ignore",invalid="ignore",divide="ignore"):
    diagnostics.run()

### Main timing, phrase and claim estimates

In [ ]:
import analysis
with np.errstate(over="ignore",invalid="ignore",divide="ignore"):
    analysis.run()

### Repeatability and measurement sensitivity

In [ ]:
import measurement_checks
with np.errstate(over="ignore",invalid="ignore",divide="ignore"):
    measurement_checks.run()

### Annual, market, macro and follower comparisons

In [ ]:
import supplementary
with np.errstate(over="ignore",invalid="ignore",divide="ignore"):
    supplementary.run()

### Lexical ties and discretization

In [ ]:
import granularity_ties
with np.errstate(over="ignore",invalid="ignore",divide="ignore"):
    granularity_ties.run()

### Attention during historical windows

In [ ]:
import attention_events
with np.errstate(over="ignore",invalid="ignore",divide="ignore"):
    attention_events.run()

### Measurement agreement and paired method comparisons

In [ ]:
import representation_diagnostics
with np.errstate(over="ignore",invalid="ignore",divide="ignore"):
    representation_diagnostics.run()

### Clustering sensitivity

In [ ]:
import inference_sensitivity
with np.errstate(over="ignore",invalid="ignore",divide="ignore"):
    inference_sensitivity.run()

### Common-news counterexample

In [ ]:
import identification
with np.errstate(over="ignore",invalid="ignore",divide="ignore"):
    identification.run()

### Reconstruction and completed scoring checks

In [ ]:
import saved_measurements
with np.errstate(over="ignore",invalid="ignore",divide="ignore"):
    saved_measurements.leader_audit()
    saved_measurements.reconstruction_summary()

### Independent statistical checks

In [ ]:
import verification
with np.errstate(over="ignore",invalid="ignore",divide="ignore"):
    verification.run()

In [ ]:
import publication
publication.configure(PACKAGE)
TABLE_DATA=publication.TABLE_DATA
publication.verify_text_values()
publication.verify_appendix_references()
print("Text values agree with fresh calculations. The manuscript sources are unchanged.")

## Main paper — exhibits in order


### Figure 1 — Annual comovement by concept.


In [ ]:
publication.show_figure('fig:annual')


### Figure 2 — Measurement and timing of the design.


In [ ]:
publication.show_figure('fig:timing')


### Table 1 — Joint timing contrast $D$ across measurements.


In [ ]:
publication.show_table('tab:methods')


### Figure 3 — Phrase-timing contrasts across windows.


In [ ]:
publication.show_figure('fig:phrase-profile')


## Online appendix — exhibits in order


### Figure A1 — Coverage and document length.


In [ ]:
publication.show_figure('fig:coverage')


### Table A1 — Jurisdiction coverage.


In [ ]:
publication.show_table('tab:coverage-2')


### Table A2 — Annual lexical projection shares by country group.


In [ ]:
publication.show_table('tab:sync')


### Figure A2 — Topic attention with and without fixed country coverage.


In [ ]:
publication.show_figure('fig:attention')


### Table A3 — Preceding and following Fed changes, estimated jointly.


In [ ]:
publication.show_table('tab:placebo')


### Table A4 — Sensitivity of the lexical timing contrast.


In [ ]:
publication.show_table('tab:sensitivity')


### Table A5 — Hawkish results under different clustering choices.


In [ ]:
publication.show_table('tab:clustering')


### Table A6 — A worked example with common news and zero transmission.


In [ ]:
publication.show_table('tab:counterexample')


### Figure A3 — Phrase-timing contrasts by Fed mention and source language.


In [ ]:
publication.show_figure('fig:phrase-subgroups')


### Table A7 — Directional-claim alignment across windows and draws.


In [ ]:
publication.show_table('tab:claims')


### Table A8 — How consistent repeated ratings are, by concept.


In [ ]:
publication.show_table('tab:reliability')


### Table A9 — Testing whether leader-measurement methods actually differ.


In [ ]:
publication.show_table('tab:coefficientdifferences')


### Table A10 — How well the reconstruction recovers the original labels.


In [ ]:
publication.show_table('tab:reconstruction')
